In [ ]:
from pathlib import Path
import sys
import os

# Detect project root / Detecta a raiz do projeto
CURRENT_DIR = Path.cwd()

# If running from notebooks folder, move one level up / Se estiver rodando da pasta notebooks, sobe um nível
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

# Add project root to Python path / Adiciona a raiz do projeto ao caminho do Python
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PYTHON:", sys.executable)
print("CWD:", Path.cwd())

In [ ]:
from dotenv import load_dotenv
import os

# Load environment variables from .env / Carrega as variáveis de ambiente do .env
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Import project configuration / Importa as configurações do projeto
from src.config import (
    validate_env,
    GROQ_MODEL_AGENT,
    GROQ_MODEL_FAST,
    GROQ_MODEL_REASONING,
    GROQ_MODEL_GENERAL,
    LANGSMITH_PROJECT,
    LANGFLOW_BASE_URL,
)

# Validate required environment variables / Valida as variáveis de ambiente obrigatórias
env_status = validate_env()

print("Environment status / Status do ambiente:")
for key, value in env_status.items():
    print(f"{key}: {value}")

print("\nGroq models / Modelos Groq:")
print("Agent model / Modelo do agente:", GROQ_MODEL_AGENT)
print("Fast model / Modelo rápido:", GROQ_MODEL_FAST)
print("Reasoning model / Modelo de raciocínio:", GROQ_MODEL_REASONING)
print("General model / Modelo geral:", GROQ_MODEL_GENERAL)

print("\nLangSmith / LangSmith:")
print("LangSmith project / Projeto LangSmith:", LANGSMITH_PROJECT)
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGCHAIN_TRACING_V2:", os.getenv("LANGCHAIN_TRACING_V2"))

print("\nLangflow / Langflow:")
print("Langflow URL:", LANGFLOW_BASE_URL)
print("Langflow API key:", "OK" if os.getenv("LANGFLOW_API_KEY") else "NÃO ENCONTRADA")

In [ ]:
from langchain_groq import ChatGroq

# Create fast LLM for simple and low-cost tasks / Cria o LLM rápido para tarefas simples e de baixo custo
llm_fast = ChatGroq(
    model=GROQ_MODEL_FAST,
    temperature=0
)

# Create agent LLM for orchestration, structured decisions and JSON outputs / Cria o LLM do agente para orquestração, decisões estruturadas e saídas JSON
llm_agent = ChatGroq(
    model=GROQ_MODEL_AGENT,
    temperature=0
)

# Create reasoning LLM for complex analysis and critical decisions / Cria o LLM de raciocínio para análises complexas e decisões críticas
llm_reasoning = ChatGroq(
    model=GROQ_MODEL_REASONING,
    temperature=0
)

# Create general LLM for natural responses, synthesis and final communication / Cria o LLM geral para respostas naturais, síntese e comunicação final
llm_general = ChatGroq(
    model=GROQ_MODEL_GENERAL,
    temperature=0
)

print("Multi-LLM setup completed / Configuração multi-LLM concluída")
print("llm_fast:", GROQ_MODEL_FAST)
print("llm_agent:", GROQ_MODEL_AGENT)
print("llm_reasoning:", GROQ_MODEL_REASONING)
print("llm_general:", GROQ_MODEL_GENERAL)

In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers

# Define test prompts for each model / Define prompts de teste para cada modelo
model_tests = [
    {
        "name": "fast",
        "llm": llm_fast,
        "task": "Classifique em uma palavra: 'Quero automatizar meu atendimento com IA'.",
        "expected_role": "quick classification / classificação rápida",
    },
    {
        "name": "agent",
        "llm": llm_agent,
        "task": "Gere um JSON simples com intent, priority e recommended_action para: 'Quero integrar meu CRM com IA'.",
        "expected_role": "agent orchestration and JSON / orquestração do agente e JSON",
    },
    {
        "name": "reasoning",
        "llm": llm_reasoning,
        "task": "Analise em uma frase se uma solicitação ambígua deve ser revisada por humano.",
        "expected_role": "complex reasoning / raciocínio complexo",
    },
    {
        "name": "general",
        "llm": llm_general,
        "task": "Explique de forma natural e curta o que é o Sentrya Ops V2.",
        "expected_role": "natural communication / comunicação natural",
    },
]

# Execute each model test / Executa o teste de cada modelo
for test in model_tests:
    print("=" * 80)
    print(f"Testing model / Testando modelo: {test['name']}")
    print(f"Role / Função: {test['expected_role']}")

    response = test["llm"].with_config(
        {
            "run_name": f"sentrya_ops_v2_test_{test['name']}_llm",
            "tags": ["sentrya-ops-v2", "multi-llm", test["name"]],
            "metadata": {
                "model_role": test["expected_role"],
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(test["task"])

    print("Response / Resposta:")
    print(response.content)

# Wait for LangSmith traces to finish / Aguarda os traces do LangSmith finalizarem
wait_for_all_tracers()

print("MULTI_LLM_TEST_OK")

In [ ]:
from typing import TypedDict, Optional, Dict, Any

# Define the main LangGraph state / Define o estado principal do LangGraph
class SentryaOpsState(TypedDict):
    user_input: str
    intent: Optional[str]
    priority: Optional[str]
    selected_model: Optional[str]
    confidence: Optional[str]
    reasoning_required: Optional[bool]
    human_review_required: Optional[bool]
    analysis: Optional[str]
    final_response: Optional[str]
    structured_output: Optional[Dict[str, Any]]


# Create a sample initial state / Cria um estado inicial de exemplo
sample_state: SentryaOpsState = {
    "user_input": "Quero automatizar meu atendimento e integrar com meu CRM.",
    "intent": None,
    "priority": None,
    "selected_model": None,
    "confidence": None,
    "reasoning_required": None,
    "human_review_required": None,
    "analysis": None,
    "final_response": None,
    "structured_output": None,
}

print("SentryaOpsState created successfully / SentryaOpsState criado com sucesso")
print(sample_state)

In [ ]:
import json
import re

# Extract JSON from model response / Extrai JSON da resposta do modelo
def extract_json_from_text(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try to extract JSON block from markdown or raw text / Tenta extrair bloco JSON de markdown ou texto bruto
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if not match:
        raise ValueError(f"No JSON found in model response / Nenhum JSON encontrado na resposta do modelo: {text}")

    return json.loads(match.group(0))


# Classify user input with the fast LLM / Classifica a entrada do usuário com o LLM rápido
def classify_input(state: SentryaOpsState) -> dict:
    user_input = state["user_input"]

    prompt = f"""
You are the fast classification layer of Sentrya Ops V2.

Analyze the user request and return ONLY valid JSON.

User request:
{user_input}

Return this JSON schema:
{{
  "intent": "commercial_lead | support_request | technical_request | general_question | unclear",
  "priority": "low | medium | high",
  "confidence": "low | medium | high",
  "reasoning_required": true or false,
  "human_review_required": true or false,
  "selected_model": "fast | agent | reasoning | general",
  "analysis": "short explanation in Portuguese"
}}

Rules:
- Use "reasoning" when the request is complex, ambiguous, risky or requires deeper analysis.
- Use "agent" when the request needs operational decision or structured workflow.
- Use "general" when the request mainly needs a natural final response.
- Use "fast" only for simple classification or simple validation.
- Return only JSON. Do not use markdown.
"""

    response = llm_fast.with_config(
        {
            "run_name": "sentrya_ops_v2_classify_input",
            "tags": ["sentrya-ops-v2", "langgraph", "node", "classification", "fast-llm"],
            "metadata": {
                "node": "classify_input",
                "model_role": "fast classification / classificação rápida",
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(prompt)

    parsed = extract_json_from_text(response.content)

    return {
        "intent": parsed.get("intent"),
        "priority": parsed.get("priority"),
        "confidence": parsed.get("confidence"),
        "reasoning_required": parsed.get("reasoning_required"),
        "human_review_required": parsed.get("human_review_required"),
        "selected_model": parsed.get("selected_model"),
        "analysis": parsed.get("analysis"),
    }


# Test classify_input node / Testa o node classify_input
classification_result = classify_input(sample_state)

print("Classification result / Resultado da classificação:")
print(json.dumps(classification_result, indent=2, ensure_ascii=False))

In [ ]:
# Merge initial state with classification result / Mescla o estado inicial com o resultado da classificação
classified_state: SentryaOpsState = {
    **sample_state,
    **classification_result,
}

# Select the best LLM based on the classification route / Seleciona o melhor LLM com base na rota da classificação
def select_llm_by_route(state: SentryaOpsState):
    selected_model = state.get("selected_model") or "agent"

    model_router = {
        "fast": {
            "llm": llm_fast,
            "model_name": GROQ_MODEL_FAST,
            "role": "fast classification and validation / classificação rápida e validação",
        },
        "agent": {
            "llm": llm_agent,
            "model_name": GROQ_MODEL_AGENT,
            "role": "agent orchestration and structured decision / orquestração do agente e decisão estruturada",
        },
        "reasoning": {
            "llm": llm_reasoning,
            "model_name": GROQ_MODEL_REASONING,
            "role": "complex reasoning and critical analysis / raciocínio complexo e análise crítica",
        },
        "general": {
            "llm": llm_general,
            "model_name": GROQ_MODEL_GENERAL,
            "role": "natural response and synthesis / resposta natural e síntese",
        },
    }

    return model_router.get(selected_model, model_router["agent"])


# Test model routing / Testa o roteamento de modelo
selected_route = select_llm_by_route(classified_state)

print("Selected route / Rota selecionada:")
print("selected_model:", classified_state.get("selected_model"))
print("model_name:", selected_route["model_name"])
print("role:", selected_route["role"])

In [ ]:
# Generate structured response using the selected model / Gera resposta estruturada usando o modelo selecionado
def generate_structured_response(state: SentryaOpsState) -> dict:
    selected_route = select_llm_by_route(state)

    llm = selected_route["llm"]
    model_name = selected_route["model_name"]
    model_role = selected_route["role"]

    prompt = f"""
You are Sentrya Ops V2, an Agentic AI Operations Desk.

Your task is to generate a structured operational response based on the classified request.

User request:
{state["user_input"]}

Classification:
intent: {state.get("intent")}
priority: {state.get("priority")}
confidence: {state.get("confidence")}
reasoning_required: {state.get("reasoning_required")}
human_review_required: {state.get("human_review_required")}
analysis: {state.get("analysis")}

Selected model:
{model_name}

Return ONLY valid JSON with this schema:
{{
  "final_response": "short natural response in Portuguese",
  "structured_output": {{
    "status": "success",
    "intent": "{state.get("intent")}",
    "priority": "{state.get("priority")}",
    "confidence": "{state.get("confidence")}",
    "selected_model": "{state.get("selected_model")}",
    "model_name": "{model_name}",
    "recommended_action": "short recommended action",
    "requires_human_review": true or false,
    "summary": "short operational summary in Portuguese"
  }}
}}

Rules:
- Return only JSON.
- Do not use markdown.
- Keep the response concise and operational.
"""

    response = llm.with_config(
        {
            "run_name": "sentrya_ops_v2_generate_structured_response",
            "tags": ["sentrya-ops-v2", "langgraph", "node", "structured-response", state.get("selected_model")],
            "metadata": {
                "node": "generate_structured_response",
                "selected_model": state.get("selected_model"),
                "model_name": model_name,
                "model_role": model_role,
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(prompt)

    parsed = extract_json_from_text(response.content)

    return {
        "final_response": parsed.get("final_response"),
        "structured_output": parsed.get("structured_output"),
    }


# Test structured response node / Testa o node de resposta estruturada
structured_response_result = generate_structured_response(classified_state)

print("Structured response result / Resultado da resposta estruturada:")
print(json.dumps(structured_response_result, indent=2, ensure_ascii=False))

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.tracers.langchain import wait_for_all_tracers

# Create LangGraph workflow / Cria o workflow LangGraph
workflow = StateGraph(SentryaOpsState)

# Add graph nodes / Adiciona os nodes do grafo
workflow.add_node("classify_input", classify_input)
workflow.add_node("generate_structured_response", generate_structured_response)

# Define graph flow / Define o fluxo do grafo
workflow.add_edge(START, "classify_input")
workflow.add_edge("classify_input", "generate_structured_response")
workflow.add_edge("generate_structured_response", END)

# Compile graph / Compila o grafo
sentrya_graph = workflow.compile()

print("Sentrya Ops V2 LangGraph compiled successfully / LangGraph do Sentrya Ops V2 compilado com sucesso")

In [ ]:
# Create test input state / Cria o estado de entrada para teste
test_state: SentryaOpsState = {
    "user_input": "Quero automatizar meu atendimento com IA, integrar com meu CRM e organizar melhor os leads que chegam pelo site.",
    "intent": None,
    "priority": None,
    "selected_model": None,
    "confidence": None,
    "reasoning_required": None,
    "human_review_required": None,
    "analysis": None,
    "final_response": None,
    "structured_output": None,
}

# Execute Sentrya Ops V2 graph / Executa o grafo do Sentrya Ops V2
graph_result = sentrya_graph.with_config(
    {
        "run_name": "sentrya_ops_v2_first_langgraph_flow",
        "tags": ["sentrya-ops-v2", "langgraph", "first-flow", "multi-llm"],
        "metadata": {
            "source": "jupyterlab-langgraph-notebook",
            "architecture": "multi-model-routing",
        },
    }
).invoke(test_state)

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("LANGGRAPH_FIRST_FLOW_OK")
print(json.dumps(graph_result, indent=2, ensure_ascii=False))

In [ ]:
from IPython.display import Markdown, display
import json

# Build clean operational output / Cria uma saída operacional limpa
def build_clean_output(result: dict) -> dict:
    structured = result.get("structured_output") or {}

    clean_output = {
        "status": structured.get("status", "success"),
        "intent": result.get("intent"),
        "priority": result.get("priority"),
        "confidence": result.get("confidence"),
        "selected_route": result.get("selected_model"),
        "model_used": structured.get("model_name"),
        "requires_human_review": structured.get("requires_human_review"),
        "recommended_action": structured.get("recommended_action"),
        "summary": structured.get("summary"),
        "final_response": result.get("final_response"),
    }

    return clean_output


# Display clean operational output / Exibe a saída operacional limpa
def display_clean_output(result: dict) -> dict:
    clean_output = build_clean_output(result)

    markdown_output = f"""
# Sentrya Ops V2 — Operational Result

**Status:** {clean_output.get("status")}  
**Intent:** {clean_output.get("intent")}  
**Priority:** {clean_output.get("priority")}  
**Confidence:** {clean_output.get("confidence")}  
**Selected Route:** {clean_output.get("selected_route")}  
**Model Used:** {clean_output.get("model_used")}  
**Human Review Required:** {clean_output.get("requires_human_review")}  

## Recommended Action
{clean_output.get("recommended_action")}

## Summary
{clean_output.get("summary")}

## Final Response
{clean_output.get("final_response")}
"""

    display(Markdown(markdown_output))

    return clean_output


# Generate and display clean result / Gera e exibe o resultado limpo
clean_result = display_clean_output(graph_result)

print("CLEAN_OUTPUT_JSON")
print(json.dumps(clean_result, indent=2, ensure_ascii=False))

In [ ]:
# Define multiple test scenarios / Define múltiplos cenários de teste
test_cases = [
    {
        "case": "commercial_lead",
        "user_input": "Quero automatizar meu atendimento com IA e integrar os leads ao meu CRM.",
    },
    {
        "case": "technical_request",
        "user_input": "Preciso conectar um webhook do n8n com um agente LangGraph e retornar JSON estruturado.",
    },
    {
        "case": "complex_analysis",
        "user_input": "Tenho vários canais de entrada, CRM desorganizado, baixa resposta comercial e preciso decidir qual processo automatizar primeiro.",
    },
    {
        "case": "general_question",
        "user_input": "Explique de forma simples como um agente de IA pode ajudar uma operação empresarial.",
    },
]

# Run graph for each scenario / Executa o grafo para cada cenário
multi_case_results = []

for item in test_cases:
    input_state: SentryaOpsState = {
        "user_input": item["user_input"],
        "intent": None,
        "priority": None,
        "selected_model": None,
        "confidence": None,
        "reasoning_required": None,
        "human_review_required": None,
        "analysis": None,
        "final_response": None,
        "structured_output": None,
    }

    result = sentrya_graph.with_config(
        {
            "run_name": f"sentrya_ops_v2_case_{item['case']}",
            "tags": ["sentrya-ops-v2", "langgraph", "multi-case-test", item["case"]],
            "metadata": {
                "case": item["case"],
                "source": "jupyterlab-langgraph-notebook",
                "architecture": "multi-model-routing",
            },
        }
    ).invoke(input_state)

    clean = build_clean_output(result)

    multi_case_results.append({
        "case": item["case"],
        "intent": clean.get("intent"),
        "priority": clean.get("priority"),
        "confidence": clean.get("confidence"),
        "selected_route": clean.get("selected_route"),
        "model_used": clean.get("model_used"),
        "requires_human_review": clean.get("requires_human_review"),
        "recommended_action": clean.get("recommended_action"),
    })

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("MULTI_CASE_LANGGRAPH_TEST_OK")

In [ ]:
import pandas as pd

# Create results table / Cria tabela de resultados
results_df = pd.DataFrame(multi_case_results)

results_df

In [ ]:
# Refined input classifier with stronger routing rules / Classificador refinado com regras mais fortes de roteamento
def classify_input(state: SentryaOpsState) -> dict:
    user_input = state["user_input"]

    prompt = f"""
You are the fast classification and routing layer of Sentrya Ops V2.

Your job is to classify the user request and decide which model should handle the next step.

User request:
{user_input}

Return ONLY valid JSON with this schema:
{{
  "intent": "commercial_lead | support_request | technical_request | complex_analysis | general_question | unclear",
  "priority": "low | medium | high",
  "confidence": "low | medium | high",
  "reasoning_required": true or false,
  "human_review_required": true or false,
  "selected_model": "fast | agent | reasoning | general",
  "analysis": "short explanation in Portuguese"
}}

Routing rules:
- Use "agent" for commercial leads, CRM integration, automation requests, business workflow requests, API integration, n8n, Langflow, LangChain or LangGraph implementation requests.
- Use "reasoning" for complex analysis, ambiguous decisions, multiple business problems, prioritization, strategy, risk, architecture decisions or unclear high-impact requests.
- Use "general" for simple educational explanations, conceptual questions or natural-language summaries.
- Use "fast" only for very simple validation, simple classification, short yes/no checks or low-risk preprocessing.
- Do NOT use "fast" as selected_model for CRM automation, lead automation, API integration, LangGraph, n8n or operational workflow requests.
- A technical request involving n8n, webhook, API, LangGraph, JSON or agent workflow should usually be "technical_request" with selected_model "agent".
- A request mentioning many problems at once, such as channels, CRM, low response, prioritization or deciding what to automate first, should be "complex_analysis" with selected_model "reasoning".
- Return only JSON. Do not use markdown.
"""

    response = llm_fast.with_config(
        {
            "run_name": "sentrya_ops_v2_classify_input_refined",
            "tags": ["sentrya-ops-v2", "langgraph", "node", "classification", "fast-llm", "refined-routing"],
            "metadata": {
                "node": "classify_input",
                "model_role": "fast classification and routing / classificação rápida e roteamento",
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(prompt)

    parsed = extract_json_from_text(response.content)

    return {
        "intent": parsed.get("intent"),
        "priority": parsed.get("priority"),
        "confidence": parsed.get("confidence"),
        "reasoning_required": parsed.get("reasoning_required"),
        "human_review_required": parsed.get("human_review_required"),
        "selected_model": parsed.get("selected_model"),
        "analysis": parsed.get("analysis"),
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

# Rebuild LangGraph workflow with refined classifier / Reconstrói o workflow LangGraph com o classificador refinado
workflow = StateGraph(SentryaOpsState)

# Add graph nodes / Adiciona os nodes do grafo
workflow.add_node("classify_input", classify_input)
workflow.add_node("generate_structured_response", generate_structured_response)

# Define graph flow / Define o fluxo do grafo
workflow.add_edge(START, "classify_input")
workflow.add_edge("classify_input", "generate_structured_response")
workflow.add_edge("generate_structured_response", END)

# Compile graph / Compila o grafo
sentrya_graph = workflow.compile()

print("Refined Sentrya Ops V2 LangGraph compiled successfully / LangGraph refinado do Sentrya Ops V2 compilado com sucesso")

In [ ]:
# Define multiple test scenarios / Define múltiplos cenários de teste
test_cases = [
    {
        "case": "commercial_lead",
        "user_input": "Quero automatizar meu atendimento com IA e integrar os leads ao meu CRM.",
    },
    {
        "case": "technical_request",
        "user_input": "Preciso conectar um webhook do n8n com um agente LangGraph e retornar JSON estruturado.",
    },
    {
        "case": "complex_analysis",
        "user_input": "Tenho vários canais de entrada, CRM desorganizado, baixa resposta comercial e preciso decidir qual processo automatizar primeiro.",
    },
    {
        "case": "general_question",
        "user_input": "Explique de forma simples como um agente de IA pode ajudar uma operação empresarial.",
    },
]

# Run graph for each scenario / Executa o grafo para cada cenário
multi_case_results = []

for item in test_cases:
    input_state: SentryaOpsState = {
        "user_input": item["user_input"],
        "intent": None,
        "priority": None,
        "selected_model": None,
        "confidence": None,
        "reasoning_required": None,
        "human_review_required": None,
        "analysis": None,
        "final_response": None,
        "structured_output": None,
    }

    result = sentrya_graph.with_config(
        {
            "run_name": f"sentrya_ops_v2_refined_case_{item['case']}",
            "tags": ["sentrya-ops-v2", "langgraph", "refined-routing-test", item["case"]],
            "metadata": {
                "case": item["case"],
                "source": "jupyterlab-langgraph-notebook",
                "architecture": "multi-model-routing",
            },
        }
    ).invoke(input_state)

    clean = build_clean_output(result)

    multi_case_results.append({
        "case": item["case"],
        "intent": clean.get("intent"),
        "priority": clean.get("priority"),
        "confidence": clean.get("confidence"),
        "selected_route": clean.get("selected_route"),
        "model_used": clean.get("model_used"),
        "requires_human_review": clean.get("requires_human_review"),
        "recommended_action": clean.get("recommended_action"),
    })

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("REFINED_MULTI_CASE_LANGGRAPH_TEST_OK")

In [ ]:
import pandas as pd

# Create results table / Cria tabela de resultados
results_df = pd.DataFrame(multi_case_results)

results_df

In [ ]:
# Refined input classifier with stronger general-question detection / Classificador refinado com melhor detecção de perguntas gerais
def classify_input(state: SentryaOpsState) -> dict:
    user_input = state["user_input"]

    prompt = f"""
You are the fast classification and routing layer of Sentrya Ops V2.

Your job is to classify the user request and decide which model should handle the next step.

User request:
{user_input}

Return ONLY valid JSON with this schema:
{{
  "intent": "commercial_lead | support_request | technical_request | complex_analysis | general_question | unclear",
  "priority": "low | medium | high",
  "confidence": "low | medium | high",
  "reasoning_required": true or false,
  "human_review_required": true or false,
  "selected_model": "fast | agent | reasoning | general",
  "analysis": "short explanation in Portuguese"
}}

Core routing rules:
- Use "general_question" with selected_model "general" when the user asks for a simple explanation, concept, definition, summary, educational answer, or asks "what is", "explain", "how does it work", "de forma simples", "explique", "o que é", or "como funciona".
- Use "technical_request" with selected_model "agent" when the user asks to build, configure, implement, connect, integrate, debug, deploy, create a workflow, call an API, return JSON, use n8n, LangGraph, LangChain, Langflow, webhook or automation logic.
- Use "commercial_lead" with selected_model "agent" when the user shows interest in buying, hiring, automating their business, CRM, leads, sales process, customer service or business operations.
- Use "complex_analysis" with selected_model "reasoning" when the user presents multiple problems, strategic decisions, prioritization, risk, architecture decisions, ambiguous business context or high-impact analysis.
- Use "fast" only for very simple validation, simple classification, yes/no checks or low-risk preprocessing.

Important distinction:
- If the user only wants to understand or learn, route to "general".
- If the user wants the system to do, build, connect, automate, configure or decide operationally, route to "agent" or "reasoning".
- Do NOT classify educational explanation requests as technical_request.
- Do NOT use "agent" for simple conceptual explanations.

Examples:
- "Explique de forma simples como um agente de IA ajuda uma empresa" -> intent "general_question", selected_model "general"
- "O que é LangGraph?" -> intent "general_question", selected_model "general"
- "Como funciona um webhook?" -> intent "general_question", selected_model "general"
- "Preciso conectar um webhook do n8n com LangGraph" -> intent "technical_request", selected_model "agent"
- "Quero automatizar meu atendimento e integrar com CRM" -> intent "commercial_lead", selected_model "agent"
- "Tenho CRM desorganizado, vários canais e preciso decidir o que automatizar primeiro" -> intent "complex_analysis", selected_model "reasoning"

Return only JSON. Do not use markdown.
"""

    response = llm_fast.with_config(
        {
            "run_name": "sentrya_ops_v2_classify_input_general_refined",
            "tags": ["sentrya-ops-v2", "langgraph", "node", "classification", "general-refinement"],
            "metadata": {
                "node": "classify_input",
                "model_role": "fast classification and routing / classificação rápida e roteamento",
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(prompt)

    parsed = extract_json_from_text(response.content)

    return {
        "intent": parsed.get("intent"),
        "priority": parsed.get("priority"),
        "confidence": parsed.get("confidence"),
        "reasoning_required": parsed.get("reasoning_required"),
        "human_review_required": parsed.get("human_review_required"),
        "selected_model": parsed.get("selected_model"),
        "analysis": parsed.get("analysis"),
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

# Rebuild LangGraph workflow with general-question refinement / Reconstrói o workflow LangGraph com refinamento de perguntas gerais
workflow = StateGraph(SentryaOpsState)

# Add graph nodes / Adiciona os nodes do grafo
workflow.add_node("classify_input", classify_input)
workflow.add_node("generate_structured_response", generate_structured_response)

# Define graph flow / Define o fluxo do grafo
workflow.add_edge(START, "classify_input")
workflow.add_edge("classify_input", "generate_structured_response")
workflow.add_edge("generate_structured_response", END)

# Compile graph / Compila o grafo
sentrya_graph = workflow.compile()

print("General-refined Sentrya Ops V2 LangGraph compiled successfully / LangGraph do Sentrya Ops V2 refinado para perguntas gerais compilado com sucesso")

In [ ]:
# Define multiple test scenarios / Define múltiplos cenários de teste
test_cases = [
    {
        "case": "commercial_lead",
        "user_input": "Quero automatizar meu atendimento com IA e integrar os leads ao meu CRM.",
    },
    {
        "case": "technical_request",
        "user_input": "Preciso conectar um webhook do n8n com um agente LangGraph e retornar JSON estruturado.",
    },
    {
        "case": "complex_analysis",
        "user_input": "Tenho vários canais de entrada, CRM desorganizado, baixa resposta comercial e preciso decidir qual processo automatizar primeiro.",
    },
    {
        "case": "general_question",
        "user_input": "Explique de forma simples como um agente de IA pode ajudar uma operação empresarial.",
    },
]

# Run graph for each scenario / Executa o grafo para cada cenário
multi_case_results = []

for item in test_cases:
    input_state: SentryaOpsState = {
        "user_input": item["user_input"],
        "intent": None,
        "priority": None,
        "selected_model": None,
        "confidence": None,
        "reasoning_required": None,
        "human_review_required": None,
        "analysis": None,
        "final_response": None,
        "structured_output": None,
    }

    result = sentrya_graph.with_config(
        {
            "run_name": f"sentrya_ops_v2_refined_case_{item['case']}",
            "tags": ["sentrya-ops-v2", "langgraph", "refined-routing-test", item["case"]],
            "metadata": {
                "case": item["case"],
                "source": "jupyterlab-langgraph-notebook",
                "architecture": "multi-model-routing",
            },
        }
    ).invoke(input_state)

    clean = build_clean_output(result)

    multi_case_results.append({
        "case": item["case"],
        "intent": clean.get("intent"),
        "priority": clean.get("priority"),
        "confidence": clean.get("confidence"),
        "selected_route": clean.get("selected_route"),
        "model_used": clean.get("model_used"),
        "requires_human_review": clean.get("requires_human_review"),
        "recommended_action": clean.get("recommended_action"),
    })

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("REFINED_MULTI_CASE_LANGGRAPH_TEST_OK")

In [ ]:
import pandas as pd

# Create results table / Cria tabela de resultados
results_df = pd.DataFrame(multi_case_results)

results_df

In [ ]:
# Refined structured response generator by intent type / Gerador refinado de resposta estruturada por tipo de intenção
def generate_structured_response(state: SentryaOpsState) -> dict:
    selected_route = select_llm_by_route(state)

    llm = selected_route["llm"]
    model_name = selected_route["model_name"]
    model_role = selected_route["role"]

    intent = state.get("intent")
    priority = state.get("priority")
    confidence = state.get("confidence")
    selected_model = state.get("selected_model")
    human_review_required = state.get("human_review_required")
    analysis = state.get("analysis")

    prompt = f"""
You are Sentrya Ops V2, an Agentic AI Operations Desk.

Your task is to generate a clean structured operational response based on the classified request.

User request:
{state["user_input"]}

Classification:
intent: {intent}
priority: {priority}
confidence: {confidence}
selected_model: {selected_model}
reasoning_required: {state.get("reasoning_required")}
human_review_required: {human_review_required}
analysis: {analysis}

Selected model:
{model_name}

Model role:
{model_role}

Return ONLY valid JSON with this schema:
{{
  "final_response": "short natural response in Portuguese",
  "structured_output": {{
    "status": "success",
    "intent": "{intent}",
    "priority": "{priority}",
    "confidence": "{confidence}",
    "selected_model": "{selected_model}",
    "model_name": "{model_name}",
    "recommended_action": "short action aligned with the intent",
    "requires_human_review": true or false,
    "summary": "short operational summary in Portuguese"
  }}
}}

Response rules by intent:
- If intent is "commercial_lead", recommend a discovery call, requirement mapping, CRM/process diagnosis or commercial qualification.
- If intent is "technical_request", recommend a technical implementation step, integration plan, API/webhook validation or structured development action.
- If intent is "complex_analysis", recommend diagnosis, prioritization, process mapping, risk review or strategic analysis before implementation.
- If intent is "general_question", provide a simple educational explanation and recommend learning/understanding, NOT implementation.
- If intent is "support_request", recommend support triage, issue reproduction, logs review or troubleshooting.
- If intent is "unclear", recommend asking for more context before deciding the next step.

Important:
- Do not recommend implementation for "general_question" unless the user explicitly asks to build or configure something.
- Keep the final response concise, useful and human.
- Keep the structured_output operational and clean.
- Return only JSON.
- Do not use markdown.
"""

    response = llm.with_config(
        {
            "run_name": "sentrya_ops_v2_generate_structured_response_refined",
            "tags": [
                "sentrya-ops-v2",
                "langgraph",
                "node",
                "structured-response",
                "intent-aware-output",
                selected_model,
            ],
            "metadata": {
                "node": "generate_structured_response",
                "intent": intent,
                "selected_model": selected_model,
                "model_name": model_name,
                "model_role": model_role,
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(prompt)

    parsed = extract_json_from_text(response.content)

    return {
        "final_response": parsed.get("final_response"),
        "structured_output": parsed.get("structured_output"),
    }

In [ ]:
from langgraph.graph import StateGraph, START, END

# Rebuild LangGraph workflow with refined structured response / Reconstrói o workflow LangGraph com resposta estruturada refinada
workflow = StateGraph(SentryaOpsState)

# Add graph nodes / Adiciona os nodes do grafo
workflow.add_node("classify_input", classify_input)
workflow.add_node("generate_structured_response", generate_structured_response)

# Define graph flow / Define o fluxo do grafo
workflow.add_edge(START, "classify_input")
workflow.add_edge("classify_input", "generate_structured_response")
workflow.add_edge("generate_structured_response", END)

# Compile graph / Compila o grafo
sentrya_graph = workflow.compile()

print("Intent-aware Sentrya Ops V2 LangGraph compiled successfully / LangGraph do Sentrya Ops V2 com resposta por intenção compilado com sucesso")

In [ ]:
import pandas as pd

# Create results table / Cria tabela de resultados
results_df = pd.DataFrame(multi_case_results)

results_df

In [ ]:
# Normalize clean output for better display / Normaliza a saída limpa para melhor visualização
def normalize_clean_output(clean_output: dict) -> dict:
    intent = clean_output.get("intent")

    action_by_intent = {
        "commercial_lead": "Mapear requisitos, entender o processo atual e qualificar a oportunidade comercial.",
        "technical_request": "Validar requisitos técnicos, integração, webhook/API e estrutura do fluxo.",
        "complex_analysis": "Realizar diagnóstico, priorizar processos e definir a primeira automação com maior impacto.",
        "general_question": "Fornecer uma explicação simples e educativa sobre o conceito solicitado.",
        "support_request": "Reproduzir o problema, analisar logs e orientar o troubleshooting.",
        "unclear": "Solicitar mais contexto antes de definir a próxima ação.",
    }

    clean_output["recommended_action"] = action_by_intent.get(
        intent,
        clean_output.get("recommended_action")
    )

    clean_output["requires_human_review"] = (
        "Yes / Sim" if clean_output.get("requires_human_review") else "No / Não"
    )

    return clean_output


# Rebuild normalized table results / Reconstrói os resultados normalizados da tabela
normalized_case_results = []

for item in test_cases:
    input_state: SentryaOpsState = {
        "user_input": item["user_input"],
        "intent": None,
        "priority": None,
        "selected_model": None,
        "confidence": None,
        "reasoning_required": None,
        "human_review_required": None,
        "analysis": None,
        "final_response": None,
        "structured_output": None,
    }

    result = sentrya_graph.with_config(
        {
            "run_name": f"sentrya_ops_v2_normalized_case_{item['case']}",
            "tags": ["sentrya-ops-v2", "langgraph", "normalized-output", item["case"]],
            "metadata": {
                "case": item["case"],
                "source": "jupyterlab-langgraph-notebook",
                "architecture": "multi-model-routing",
            },
        }
    ).invoke(input_state)

    clean = build_clean_output(result)
    clean = normalize_clean_output(clean)

    normalized_case_results.append({
        "case": item["case"],
        "intent": clean.get("intent"),
        "priority": clean.get("priority"),
        "confidence": clean.get("confidence"),
        "selected_route": clean.get("selected_route"),
        "model_used": clean.get("model_used"),
        "requires_human_review": clean.get("requires_human_review"),
        "recommended_action": clean.get("recommended_action"),
    })

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("NORMALIZED_OUTPUT_TABLE_OK")

In [ ]:
import pandas as pd

# Create normalized results table / Cria tabela normalizada de resultados
normalized_df = pd.DataFrame(normalized_case_results)

# Display styled table / Exibe tabela estilizada
styled_df = (
    normalized_df.style
    .set_properties(
        subset=[
            "case",
            "intent",
            "priority",
            "confidence",
            "selected_route",
            "model_used",
            "requires_human_review",
        ],
        **{
            "text-align": "center",
            "vertical-align": "middle",
        }
    )
    .set_properties(
        subset=["recommended_action"],
        **{
            "text-align": "left",
            "white-space": "normal",
            "vertical-align": "middle",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                ],
            }
        ]
    )
)

styled_df

In [ ]:
import pandas as pd

# Create readable labels for table values / Cria rótulos legíveis para os valores da tabela
intent_labels = {
    "commercial_lead": "Lead comercial",
    "technical_request": "Solicitação técnica",
    "complex_analysis": "Análise complexa",
    "general_question": "Pergunta geral",
    "support_request": "Suporte",
    "unclear": "Indefinido",
}

priority_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

confidence_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

route_labels = {
    "fast": "Rápido",
    "agent": "Agente",
    "reasoning": "Raciocínio",
    "general": "Geral",
}

case_labels = {
    "commercial_lead": "Lead comercial",
    "technical_request": "Solicitação técnica",
    "complex_analysis": "Análise complexa",
    "general_question": "Pergunta geral",
}

# Create formatted dataframe / Cria DataFrame formatado
display_df = pd.DataFrame(normalized_case_results).copy()

# Apply readable labels / Aplica rótulos legíveis
display_df["case"] = display_df["case"].map(case_labels).fillna(display_df["case"])
display_df["intent"] = display_df["intent"].map(intent_labels).fillna(display_df["intent"])
display_df["priority"] = display_df["priority"].map(priority_labels).fillna(display_df["priority"])
display_df["confidence"] = display_df["confidence"].map(confidence_labels).fillna(display_df["confidence"])
display_df["selected_route"] = display_df["selected_route"].map(route_labels).fillna(display_df["selected_route"])

# Rename columns for better readability / Renomeia colunas para melhor leitura
display_df = display_df.rename(
    columns={
        "case": "Cenário",
        "intent": "Intenção",
        "priority": "Prioridade",
        "confidence": "Confiança",
        "selected_route": "Rota LLM",
        "model_used": "Modelo Usado",
        "requires_human_review": "Revisão Humana",
        "recommended_action": "Ação Recomendada",
    }
)

# Display styled table with borders and separation lines / Exibe tabela estilizada com bordas e linhas de separação
styled_table = (
    display_df.style
    .hide(axis="index")
    .set_properties(
        **{
            "border": "1px solid #4b5563",
            "padding": "10px",
            "vertical-align": "middle",
            "font-size": "13px",
        }
    )
    .set_properties(
        subset=[
            "Cenário",
            "Intenção",
            "Prioridade",
            "Confiança",
            "Rota LLM",
            "Modelo Usado",
            "Revisão Humana",
        ],
        **{
            "text-align": "center",
            "white-space": "normal",
        }
    )
    .set_properties(
        subset=["Ação Recomendada"],
        **{
            "text-align": "left",
            "white-space": "normal",
            "min-width": "280px",
            "max-width": "360px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("padding", "12px"),
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                    ("font-weight", "bold"),
                    ("background-color", "#111827"),
                    ("color", "#f9fafb"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("border-bottom", "2px solid #374151"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                ],
            },
        ]
    )
)

styled_table

In [ ]:
import pandas as pd

# Create readable labels for scenario names / Cria rótulos legíveis para os nomes dos cenários
case_labels = {
    "commercial_lead": "Entrada comercial",
    "technical_request": "Integração técnica",
    "complex_analysis": "Diagnóstico operacional",
    "general_question": "Explicação conceitual",
}

# Create readable labels for detected intents / Cria rótulos legíveis para as intenções detectadas
intent_labels = {
    "commercial_lead": "Lead comercial",
    "technical_request": "Solicitação técnica",
    "complex_analysis": "Análise complexa",
    "general_question": "Pergunta geral",
    "support_request": "Suporte",
    "unclear": "Indefinido",
}

# Create readable labels for priority values / Cria rótulos legíveis para os valores de prioridade
priority_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

# Create readable labels for confidence values / Cria rótulos legíveis para os valores de confiança
confidence_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

# Create readable labels for LLM routes / Cria rótulos legíveis para as rotas dos LLMs
route_labels = {
    "fast": "Rápido",
    "agent": "Agente",
    "reasoning": "Raciocínio",
    "general": "Geral",
}

# Create formatted dataframe from normalized results / Cria DataFrame formatado a partir dos resultados normalizados
display_df = pd.DataFrame(normalized_case_results).copy()

# Apply readable labels to each column / Aplica rótulos legíveis em cada coluna
display_df["case"] = display_df["case"].map(case_labels).fillna(display_df["case"])
display_df["intent"] = display_df["intent"].map(intent_labels).fillna(display_df["intent"])
display_df["priority"] = display_df["priority"].map(priority_labels).fillna(display_df["priority"])
display_df["confidence"] = display_df["confidence"].map(confidence_labels).fillna(display_df["confidence"])
display_df["selected_route"] = display_df["selected_route"].map(route_labels).fillna(display_df["selected_route"])

# Rename columns for better readability / Renomeia colunas para melhor leitura
display_df = display_df.rename(
    columns={
        "case": "Cenário de Teste",
        "intent": "Intenção Detectada",
        "priority": "Prioridade",
        "confidence": "Confiança",
        "selected_route": "Rota LLM",
        "model_used": "Modelo Usado",
        "requires_human_review": "Revisão Humana",
        "recommended_action": "Ação Recomendada",
    }
)

# Reorder columns for better reading / Reordena as colunas para melhor leitura
display_df = display_df[
    [
        "Cenário de Teste",
        "Intenção Detectada",
        "Prioridade",
        "Confiança",
        "Rota LLM",
        "Modelo Usado",
        "Revisão Humana",
        "Ação Recomendada",
    ]
]

# Display styled table with borders and separation lines / Exibe tabela estilizada com bordas e linhas de separação
styled_table = (
    display_df.style
    .hide(axis="index")
    .set_properties(
        **{
            "border": "1px solid #4b5563",
            "padding": "10px",
            "vertical-align": "middle",
            "font-size": "13px",
        }
    )
    .set_properties(
        subset=[
            "Cenário de Teste",
            "Intenção Detectada",
            "Prioridade",
            "Confiança",
            "Rota LLM",
            "Modelo Usado",
            "Revisão Humana",
        ],
        **{
            "text-align": "center",
            "white-space": "normal",
            "vertical-align": "middle",
        }
    )
    .set_properties(
        subset=["Ação Recomendada"],
        **{
            "text-align": "left",
            "white-space": "normal",
            "vertical-align": "middle",
            "min-width": "300px",
            "max-width": "420px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("padding", "12px"),
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                    ("font-weight", "bold"),
                    ("background-color", "#111827"),
                    ("color", "#f9fafb"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("border-bottom", "2px solid #374151"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                ],
            },
        ]
    )
)

styled_table

In [ ]:
from pathlib import Path

# Create output folder / Cria a pasta de saída
output_dir = PROJECT_ROOT / "docs" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Save table as CSV / Salva a tabela como CSV
csv_path = output_dir / "sentrya_ops_v2_multi_llm_routing_table.csv"
display_df.to_csv(csv_path, index=False, encoding="utf-8")

# Save table as HTML / Salva a tabela como HTML
html_path = output_dir / "sentrya_ops_v2_multi_llm_routing_table.html"
styled_table.to_html(html_path)

print("Table saved successfully / Tabela salva com sucesso")
print("CSV:", csv_path)
print("HTML:", html_path)

In [ ]:
import time

# Extract token usage from LLM response / Extrai o uso de tokens da resposta do LLM
def extract_token_usage(response) -> dict:
    usage_metadata = getattr(response, "usage_metadata", None) or {}
    response_metadata = getattr(response, "response_metadata", None) or {}

    if usage_metadata:
        input_tokens = usage_metadata.get("input_tokens", 0) or 0
        output_tokens = usage_metadata.get("output_tokens", 0) or 0
        total_tokens = usage_metadata.get("total_tokens", input_tokens + output_tokens) or 0
    else:
        token_usage = response_metadata.get("token_usage", {}) or {}
        input_tokens = token_usage.get("prompt_tokens", token_usage.get("input_tokens", 0)) or 0
        output_tokens = token_usage.get("completion_tokens", token_usage.get("output_tokens", 0)) or 0
        total_tokens = token_usage.get("total_tokens", input_tokens + output_tokens) or 0

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
    }


# Merge token metrics from multiple LLM calls / Consolida métricas de tokens de múltiplas chamadas LLM
def merge_metrics(current_metrics: dict | None, new_usage: dict, model_name: str) -> dict:
    current_metrics = current_metrics or {
        "llm_calls": 0,
        "input_tokens": 0,
        "output_tokens": 0,
        "total_tokens": 0,
        "models_called": [],
    }

    current_metrics["llm_calls"] += 1
    current_metrics["input_tokens"] += new_usage.get("input_tokens", 0)
    current_metrics["output_tokens"] += new_usage.get("output_tokens", 0)
    current_metrics["total_tokens"] += new_usage.get("total_tokens", 0)

    if model_name not in current_metrics["models_called"]:
        current_metrics["models_called"].append(model_name)

    current_metrics["avg_tokens_per_call"] = round(
        current_metrics["total_tokens"] / current_metrics["llm_calls"],
        2
    )

    return current_metrics


print("Metrics helpers created successfully / Funções auxiliares de métricas criadas com sucesso")

In [ ]:
# Validate metrics helper functions / Valida as funções auxiliares de métricas
print("extract_token_usage:", callable(extract_token_usage))
print("merge_metrics:", callable(merge_metrics))

In [ ]:
from typing import TypedDict, Optional, Dict, Any

# Define the main LangGraph state with metrics / Define o estado principal do LangGraph com métricas
class SentryaOpsState(TypedDict):
    user_input: str
    intent: Optional[str]
    priority: Optional[str]
    selected_model: Optional[str]
    confidence: Optional[str]
    reasoning_required: Optional[bool]
    human_review_required: Optional[bool]
    analysis: Optional[str]
    final_response: Optional[str]
    structured_output: Optional[Dict[str, Any]]
    metrics: Optional[Dict[str, Any]]


print("SentryaOpsState with metrics created successfully / SentryaOpsState com métricas criado com sucesso")

In [ ]:
# Classify input and collect token metrics / Classifica a entrada e coleta métricas de tokens
def classify_input(state: SentryaOpsState) -> dict:
    user_input = state["user_input"]

    prompt = f"""
You are the fast classification and routing layer of Sentrya Ops V2.

Your job is to classify the user request and decide which model should handle the next step.

User request:
{user_input}

Return ONLY valid JSON with this schema:
{{
  "intent": "commercial_lead | support_request | technical_request | complex_analysis | general_question | unclear",
  "priority": "low | medium | high",
  "confidence": "low | medium | high",
  "reasoning_required": true or false,
  "human_review_required": true or false,
  "selected_model": "fast | agent | reasoning | general",
  "analysis": "short explanation in Portuguese"
}}

Core routing rules:
- Use "general_question" with selected_model "general" when the user asks for a simple explanation, concept, definition, summary, educational answer, or asks "what is", "explain", "how does it work", "de forma simples", "explique", "o que é", or "como funciona".
- Use "technical_request" with selected_model "agent" when the user asks to build, configure, implement, connect, integrate, debug, deploy, create a workflow, call an API, return JSON, use n8n, LangGraph, LangChain, Langflow, webhook or automation logic.
- Use "commercial_lead" with selected_model "agent" when the user shows interest in buying, hiring, automating their business, CRM, leads, sales process, customer service or business operations.
- Use "complex_analysis" with selected_model "reasoning" when the user presents multiple problems, strategic decisions, prioritization, risk, architecture decisions, ambiguous business context or high-impact analysis.
- Use "fast" only for very simple validation, simple classification, yes/no checks or low-risk preprocessing.

Important distinction:
- If the user only wants to understand or learn, route to "general".
- If the user wants the system to do, build, connect, automate, configure or decide operationally, route to "agent" or "reasoning".
- Do NOT classify educational explanation requests as technical_request.
- Do NOT use "agent" for simple conceptual explanations.

Return only JSON. Do not use markdown.
"""

    response = llm_fast.with_config(
        {
            "run_name": "sentrya_ops_v2_classify_input_with_metrics",
            "tags": ["sentrya-ops-v2", "langgraph", "metrics", "classification"],
            "metadata": {
                "node": "classify_input",
                "model_role": "fast classification and routing / classificação rápida e roteamento",
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(prompt)

    parsed = extract_json_from_text(response.content)

    # Apply routing guardrails after LLM classification / Aplica guardrails de roteamento após a classificação do LLM
    parsed = apply_routing_guardrails(user_input, parsed)

    # Extract token usage / Extrai uso de tokens
    usage = extract_token_usage(response)

    # Merge metrics / Consolida métricas
    metrics = merge_metrics(
        state.get("metrics"),
        usage,
        GROQ_MODEL_FAST
    )

    return {
        "intent": parsed.get("intent"),
        "priority": parsed.get("priority"),
        "confidence": parsed.get("confidence"),
        "reasoning_required": parsed.get("reasoning_required"),
        "human_review_required": parsed.get("human_review_required"),
        "selected_model": parsed.get("selected_model"),
        "analysis": parsed.get("analysis"),
        "metrics": metrics,
    }


print("classify_input with metrics updated successfully / classify_input com métricas atualizado com sucesso")

In [ ]:
# Generate structured response and collect token metrics / Gera resposta estruturada e coleta métricas de tokens
def generate_structured_response(state: SentryaOpsState) -> dict:
    selected_route = select_llm_by_route(state)

    llm = selected_route["llm"]
    model_name = selected_route["model_name"]
    model_role = selected_route["role"]

    intent = state.get("intent")
    priority = state.get("priority")
    confidence = state.get("confidence")
    selected_model = state.get("selected_model")
    human_review_required = state.get("human_review_required")
    analysis = state.get("analysis")

    prompt = f"""
You are Sentrya Ops V2, an Agentic AI Operations Desk.

Your task is to generate a clean structured operational response based on the classified request.

User request:
{state["user_input"]}

Classification:
intent: {intent}
priority: {priority}
confidence: {confidence}
selected_model: {selected_model}
reasoning_required: {state.get("reasoning_required")}
human_review_required: {human_review_required}
analysis: {analysis}

Selected model:
{model_name}

Model role:
{model_role}

Return ONLY valid JSON with this schema:
{{
  "final_response": "short natural response in Portuguese",
  "structured_output": {{
    "status": "success",
    "intent": "{intent}",
    "priority": "{priority}",
    "confidence": "{confidence}",
    "selected_model": "{selected_model}",
    "model_name": "{model_name}",
    "recommended_action": "short action aligned with the intent",
    "requires_human_review": true or false,
    "summary": "short operational summary in Portuguese"
  }}
}}

Response rules by intent:
- If intent is "commercial_lead", recommend a discovery call, requirement mapping, CRM/process diagnosis or commercial qualification.
- If intent is "technical_request", recommend a technical implementation step, integration plan, API/webhook validation or structured development action.
- If intent is "complex_analysis", recommend diagnosis, prioritization, process mapping, risk review or strategic analysis before implementation.
- If intent is "general_question", provide a simple educational explanation and recommend learning/understanding, NOT implementation.
- If intent is "support_request", recommend support triage, issue reproduction, logs review or troubleshooting.
- If intent is "unclear", recommend asking for more context before deciding the next step.

Important:
- Do not recommend implementation for "general_question" unless the user explicitly asks to build or configure something.
- Keep the final response concise, useful and human.
- Keep the structured_output operational and clean.
- Return only JSON.
- Do not use markdown.
"""

    response = llm.with_config(
        {
            "run_name": "sentrya_ops_v2_generate_response_with_metrics",
            "tags": [
                "sentrya-ops-v2",
                "langgraph",
                "metrics",
                "structured-response",
                selected_model,
            ],
            "metadata": {
                "node": "generate_structured_response",
                "intent": intent,
                "selected_model": selected_model,
                "model_name": model_name,
                "model_role": model_role,
                "source": "jupyterlab-langgraph-notebook",
            },
        }
    ).invoke(prompt)

    parsed = extract_json_from_text(response.content)

    # Extract token usage / Extrai uso de tokens
    usage = extract_token_usage(response)

    # Merge metrics / Consolida métricas
    metrics = merge_metrics(
        state.get("metrics"),
        usage,
        model_name
    )

    return {
        "final_response": parsed.get("final_response"),
        "structured_output": parsed.get("structured_output"),
        "metrics": metrics,
    }


print("generate_structured_response with metrics updated successfully / generate_structured_response com métricas atualizado com sucesso")

In [ ]:
# Build clean operational output with metrics / Cria saída operacional limpa com métricas
def build_clean_output(result: dict) -> dict:
    structured = result.get("structured_output") or {}
    metrics = result.get("metrics") or {}

    clean_output = {
        "status": structured.get("status", "success"),
        "intent": result.get("intent"),
        "priority": result.get("priority"),
        "confidence": result.get("confidence"),
        "selected_route": result.get("selected_model"),
        "model_used": structured.get("model_name"),
        "requires_human_review": structured.get("requires_human_review"),
        "recommended_action": structured.get("recommended_action"),
        "summary": structured.get("summary"),
        "final_response": result.get("final_response"),
        "llm_calls": metrics.get("llm_calls"),
        "input_tokens": metrics.get("input_tokens"),
        "output_tokens": metrics.get("output_tokens"),
        "total_tokens": metrics.get("total_tokens"),
        "avg_tokens_per_call": metrics.get("avg_tokens_per_call"),
        "models_called": ", ".join(metrics.get("models_called", [])),
    }

    return clean_output


print("build_clean_output with metrics updated successfully / build_clean_output com métricas atualizado com sucesso")

In [ ]:
from langgraph.graph import StateGraph, START, END

# Rebuild LangGraph workflow with metrics / Reconstrói o workflow LangGraph com métricas
workflow = StateGraph(SentryaOpsState)

# Add graph nodes / Adiciona os nodes do grafo
workflow.add_node("classify_input", classify_input)
workflow.add_node("generate_structured_response", generate_structured_response)

# Define graph flow / Define o fluxo do grafo
workflow.add_edge(START, "classify_input")
workflow.add_edge("classify_input", "generate_structured_response")
workflow.add_edge("generate_structured_response", END)

# Compile graph / Compila o grafo
sentrya_graph = workflow.compile()

print("Sentrya Ops V2 LangGraph with metrics compiled successfully / LangGraph do Sentrya Ops V2 com métricas compilado com sucesso")

In [ ]:
# Define multiple test scenarios / Define múltiplos cenários de teste
test_cases = [
    {
        "case": "commercial_lead",
        "user_input": "Quero automatizar meu atendimento com IA e integrar os leads ao meu CRM.",
    },
    {
        "case": "technical_request",
        "user_input": "Preciso conectar um webhook do n8n com um agente LangGraph e retornar JSON estruturado.",
    },
    {
        "case": "complex_analysis",
        "user_input": "Tenho vários canais de entrada, CRM desorganizado, baixa resposta comercial e preciso decidir qual processo automatizar primeiro.",
    },
    {
        "case": "general_question",
        "user_input": "Explique de forma simples como um agente de IA pode ajudar uma operação empresarial.",
    },
]

# Run graph for each scenario with metrics / Executa o grafo para cada cenário com métricas
normalized_case_results = []

for item in test_cases:
    input_state: SentryaOpsState = {
        "user_input": item["user_input"],
        "intent": None,
        "priority": None,
        "selected_model": None,
        "confidence": None,
        "reasoning_required": None,
        "human_review_required": None,
        "analysis": None,
        "final_response": None,
        "structured_output": None,
        "metrics": None,
    }

    result = sentrya_graph.with_config(
        {
            "run_name": f"sentrya_ops_v2_metrics_case_{item['case']}",
            "tags": ["sentrya-ops-v2", "langgraph", "metrics-table", item["case"]],
            "metadata": {
                "case": item["case"],
                "source": "jupyterlab-langgraph-notebook",
                "architecture": "multi-model-routing-with-metrics",
            },
        }
    ).invoke(input_state)

    clean = build_clean_output(result)
    clean = normalize_clean_output(clean)

    normalized_case_results.append({
        "case": item["case"],
        "intent": clean.get("intent"),
        "priority": clean.get("priority"),
        "confidence": clean.get("confidence"),
        "selected_route": clean.get("selected_route"),
        "model_used": clean.get("model_used"),
        "requires_human_review": clean.get("requires_human_review"),
        "recommended_action": clean.get("recommended_action"),
        "llm_calls": clean.get("llm_calls"),
        "input_tokens": clean.get("input_tokens"),
        "output_tokens": clean.get("output_tokens"),
        "total_tokens": clean.get("total_tokens"),
        "avg_tokens_per_call": clean.get("avg_tokens_per_call"),
        "models_called": clean.get("models_called"),
    })

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("METRICS_CASE_RESULTS_OK")

In [ ]:
# Apply deterministic routing guardrails / Aplica guardrails determinísticos de roteamento
def apply_routing_guardrails(user_input: str, parsed: dict) -> dict:
    text = user_input.lower()

    complex_signals = [
        "vários canais",
        "varios canais",
        "crm desorganizado",
        "baixa resposta",
        "decidir qual processo",
        "automatizar primeiro",
        "priorizar",
        "prioridade",
        "diagnóstico",
        "diagnostico",
        "múltiplos problemas",
        "multiplos problemas",
    ]

    technical_signals = [
        "webhook",
        "api",
        "json",
        "n8n",
        "langgraph",
        "langchain",
        "langflow",
        "integrar",
        "conectar",
        "configurar",
        "deploy",
    ]

    commercial_signals = [
        "quero automatizar",
        "crm",
        "leads",
        "atendimento",
        "vendas",
        "processo comercial",
        "cliente",
    ]

    general_signals = [
        "explique",
        "o que é",
        "o que e",
        "como funciona",
        "de forma simples",
        "conceito",
        "resumo",
    ]

    # Force complex analysis route / Força rota de análise complexa
    if any(signal in text for signal in complex_signals):
        parsed["intent"] = "complex_analysis"
        parsed["priority"] = "high"
        parsed["confidence"] = "high"
        parsed["reasoning_required"] = True
        parsed["human_review_required"] = False
        parsed["selected_model"] = "reasoning"
        parsed["analysis"] = (
            "A solicitação apresenta múltiplos problemas operacionais e exige priorização estratégica antes da automação."
        )
        return parsed

    # Force technical request route / Força rota de solicitação técnica
    if any(signal in text for signal in technical_signals):
        parsed["intent"] = "technical_request"
        parsed["priority"] = parsed.get("priority") or "medium"
        parsed["confidence"] = parsed.get("confidence") or "high"
        parsed["reasoning_required"] = False
        parsed["human_review_required"] = False
        parsed["selected_model"] = "agent"
        parsed["analysis"] = (
            "A solicitação envolve integração, configuração técnica ou desenvolvimento de fluxo operacional."
        )
        return parsed

    # Force commercial lead route / Força rota de lead comercial
    if any(signal in text for signal in commercial_signals) and not any(
        signal in text for signal in general_signals
    ):
        parsed["intent"] = "commercial_lead"
        parsed["priority"] = parsed.get("priority") or "medium"
        parsed["confidence"] = parsed.get("confidence") or "high"
        parsed["reasoning_required"] = False
        parsed["human_review_required"] = False
        parsed["selected_model"] = "agent"
        parsed["analysis"] = (
            "A solicitação indica interesse em automação operacional ou melhoria de processo comercial."
        )
        return parsed

    # Force general question route / Força rota de pergunta geral
    if any(signal in text for signal in general_signals):
        parsed["intent"] = "general_question"
        parsed["priority"] = "low"
        parsed["confidence"] = parsed.get("confidence") or "high"
        parsed["reasoning_required"] = False
        parsed["human_review_required"] = False
        parsed["selected_model"] = "general"
        parsed["analysis"] = (
            "A solicitação pede uma explicação conceitual ou educativa, sem ação técnica imediata."
        )
        return parsed

    return parsed


print("Routing guardrails created successfully / Guardrails de roteamento criados com sucesso")

In [ ]:
# Define multiple test scenarios / Define múltiplos cenários de teste
test_cases = [
    {
        "case": "commercial_lead",
        "user_input": "Quero automatizar meu atendimento com IA e integrar os leads ao meu CRM.",
    },
    {
        "case": "technical_request",
        "user_input": "Preciso conectar um webhook do n8n com um agente LangGraph e retornar JSON estruturado.",
    },
    {
        "case": "complex_analysis",
        "user_input": "Tenho vários canais de entrada, CRM desorganizado, baixa resposta comercial e preciso decidir qual processo automatizar primeiro.",
    },
    {
        "case": "general_question",
        "user_input": "Explique de forma simples como um agente de IA pode ajudar uma operação empresarial.",
    },
]

# Run graph for each scenario with metrics / Executa o grafo para cada cenário com métricas
normalized_case_results = []

for item in test_cases:
    input_state: SentryaOpsState = {
        "user_input": item["user_input"],
        "intent": None,
        "priority": None,
        "selected_model": None,
        "confidence": None,
        "reasoning_required": None,
        "human_review_required": None,
        "analysis": None,
        "final_response": None,
        "structured_output": None,
        "metrics": None,
    }

    result = sentrya_graph.with_config(
        {
            "run_name": f"sentrya_ops_v2_metrics_case_{item['case']}",
            "tags": ["sentrya-ops-v2", "langgraph", "metrics-table", item["case"]],
            "metadata": {
                "case": item["case"],
                "source": "jupyterlab-langgraph-notebook",
                "architecture": "multi-model-routing-with-metrics",
            },
        }
    ).invoke(input_state)

    clean = build_clean_output(result)
    clean = normalize_clean_output(clean)

    normalized_case_results.append({
        "case": item["case"],
        "intent": clean.get("intent"),
        "priority": clean.get("priority"),
        "confidence": clean.get("confidence"),
        "selected_route": clean.get("selected_route"),
        "model_used": clean.get("model_used"),
        "requires_human_review": clean.get("requires_human_review"),
        "recommended_action": clean.get("recommended_action"),
        "llm_calls": clean.get("llm_calls"),
        "input_tokens": clean.get("input_tokens"),
        "output_tokens": clean.get("output_tokens"),
        "total_tokens": clean.get("total_tokens"),
        "avg_tokens_per_call": clean.get("avg_tokens_per_call"),
        "models_called": clean.get("models_called"),
    })

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("METRICS_CASE_RESULTS_OK")

In [ ]:
import pandas as pd

# Create readable labels for scenario names / Cria rótulos legíveis para os nomes dos cenários
case_labels = {
    "commercial_lead": "Entrada comercial",
    "technical_request": "Integração técnica",
    "complex_analysis": "Diagnóstico operacional",
    "general_question": "Explicação conceitual",
}

# Create readable labels for detected intents / Cria rótulos legíveis para as intenções detectadas
intent_labels = {
    "commercial_lead": "Lead comercial",
    "technical_request": "Solicitação técnica",
    "complex_analysis": "Análise complexa",
    "general_question": "Pergunta geral",
    "support_request": "Suporte",
    "unclear": "Indefinido",
}

# Create readable labels for priority values / Cria rótulos legíveis para os valores de prioridade
priority_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

# Create readable labels for confidence values / Cria rótulos legíveis para os valores de confiança
confidence_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

# Create readable labels for LLM routes / Cria rótulos legíveis para as rotas dos LLMs
route_labels = {
    "fast": "Rápido",
    "agent": "Agente",
    "reasoning": "Raciocínio",
    "general": "Geral",
}

# Create dataframe from normalized results / Cria DataFrame a partir dos resultados normalizados
metrics_df = pd.DataFrame(normalized_case_results).copy()

# Apply readable labels / Aplica rótulos legíveis
metrics_df["case"] = metrics_df["case"].map(case_labels).fillna(metrics_df["case"])
metrics_df["intent"] = metrics_df["intent"].map(intent_labels).fillna(metrics_df["intent"])
metrics_df["priority"] = metrics_df["priority"].map(priority_labels).fillna(metrics_df["priority"])
metrics_df["confidence"] = metrics_df["confidence"].map(confidence_labels).fillna(metrics_df["confidence"])
metrics_df["selected_route"] = metrics_df["selected_route"].map(route_labels).fillna(metrics_df["selected_route"])

# Rename columns for final monitoring table / Renomeia colunas para a tabela final de monitoramento
metrics_df = metrics_df.rename(
    columns={
        "case": "Cenário de Teste",
        "intent": "Intenção Detectada",
        "priority": "Prioridade",
        "confidence": "Confiança",
        "selected_route": "Rota LLM",
        "model_used": "Modelo Executor",
        "requires_human_review": "Revisão Humana",
        "recommended_action": "Ação Recomendada",
        "llm_calls": "Chamadas LLM",
        "input_tokens": "Tokens Entrada",
        "output_tokens": "Tokens Saída",
        "total_tokens": "Tokens Totais",
        "avg_tokens_per_call": "Média Tokens/Chamada",
        "models_called": "Modelos Chamados",
    }
)

# Reorder columns / Reordena as colunas
metrics_df = metrics_df[
    [
        "Cenário de Teste",
        "Intenção Detectada",
        "Prioridade",
        "Confiança",
        "Rota LLM",
        "Modelo Executor",
        "Modelos Chamados",
        "Chamadas LLM",
        "Tokens Entrada",
        "Tokens Saída",
        "Tokens Totais",
        "Média Tokens/Chamada",
        "Revisão Humana",
        "Ação Recomendada",
    ]
]

# Display styled metrics table / Exibe tabela de métricas estilizada
metrics_styled_table = (
    metrics_df.style
    .hide(axis="index")
    .set_properties(
        **{
            "border": "1px solid #4b5563",
            "padding": "10px",
            "vertical-align": "middle",
            "font-size": "13px",
        }
    )
    .set_properties(
        subset=[
            "Cenário de Teste",
            "Intenção Detectada",
            "Prioridade",
            "Confiança",
            "Rota LLM",
            "Modelo Executor",
            "Modelos Chamados",
            "Chamadas LLM",
            "Tokens Entrada",
            "Tokens Saída",
            "Tokens Totais",
            "Média Tokens/Chamada",
            "Revisão Humana",
        ],
        **{
            "text-align": "center",
            "white-space": "normal",
            "vertical-align": "middle",
        }
    )
    .set_properties(
        subset=["Ação Recomendada"],
        **{
            "text-align": "left",
            "white-space": "normal",
            "vertical-align": "middle",
            "min-width": "320px",
            "max-width": "460px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("padding", "12px"),
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                    ("font-weight", "bold"),
                    ("background-color", "#111827"),
                    ("color", "#f9fafb"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("border-bottom", "2px solid #374151"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                ],
            },
        ]
    )
)

metrics_styled_table

In [ ]:
# Normalize clean output with scenario-aware actions / Normaliza a saída limpa com ações por cenário
def normalize_clean_output(clean_output: dict, case: str | None = None) -> dict:
    intent = clean_output.get("intent")

    action_by_case = {
        "commercial_lead": "Mapear o processo comercial atual, entender origem dos leads, CRM utilizado e oportunidades de automação.",
        "technical_request": "Validar tecnicamente o webhook, o fluxo n8n, a chamada ao agente LangGraph e o retorno em JSON estruturado.",
        "complex_analysis": "Realizar diagnóstico dos canais de entrada, mapear gargalos no CRM e priorizar a automação por impacto, esforço e risco.",
        "general_question": "Fornecer uma explicação simples e educativa sobre como agentes de IA podem apoiar operações empresariais.",
    }

    action_by_intent = {
        "commercial_lead": "Mapear requisitos, entender o processo atual e qualificar a oportunidade comercial.",
        "technical_request": "Validar requisitos técnicos, integração, webhook/API e estrutura do fluxo.",
        "complex_analysis": "Realizar diagnóstico, priorizar processos e definir a primeira automação com maior impacto.",
        "general_question": "Fornecer uma explicação simples e educativa sobre o conceito solicitado.",
        "support_request": "Reproduzir o problema, analisar logs e orientar o troubleshooting.",
        "unclear": "Solicitar mais contexto antes de definir a próxima ação.",
    }

    clean_output["recommended_action"] = action_by_case.get(
        case,
        action_by_intent.get(intent, clean_output.get("recommended_action"))
    )

    clean_output["requires_human_review"] = (
        "Yes / Sim" if clean_output.get("requires_human_review") else "No / Não"
    )

    return clean_output


print("Scenario-aware normalization updated / Normalização por cenário atualizada")

In [ ]:
# Run graph for each scenario with metrics and scenario-aware actions / Executa o grafo com métricas e ações por cenário
normalized_case_results = []

for item in test_cases:
    input_state: SentryaOpsState = {
        "user_input": item["user_input"],
        "intent": None,
        "priority": None,
        "selected_model": None,
        "confidence": None,
        "reasoning_required": None,
        "human_review_required": None,
        "analysis": None,
        "final_response": None,
        "structured_output": None,
        "metrics": None,
    }

    result = sentrya_graph.with_config(
        {
            "run_name": f"sentrya_ops_v2_metrics_case_fixed_action_{item['case']}",
            "tags": ["sentrya-ops-v2", "langgraph", "metrics-table", "fixed-action", item["case"]],
            "metadata": {
                "case": item["case"],
                "source": "jupyterlab-langgraph-notebook",
                "architecture": "multi-model-routing-with-metrics",
            },
        }
    ).invoke(input_state)

    clean = build_clean_output(result)
    clean = normalize_clean_output(clean, item["case"])

    normalized_case_results.append({
        "case": item["case"],
        "intent": clean.get("intent"),
        "priority": clean.get("priority"),
        "confidence": clean.get("confidence"),
        "selected_route": clean.get("selected_route"),
        "model_used": clean.get("model_used"),
        "requires_human_review": clean.get("requires_human_review"),
        "recommended_action": clean.get("recommended_action"),
        "llm_calls": clean.get("llm_calls"),
        "input_tokens": clean.get("input_tokens"),
        "output_tokens": clean.get("output_tokens"),
        "total_tokens": clean.get("total_tokens"),
        "avg_tokens_per_call": clean.get("avg_tokens_per_call"),
        "models_called": clean.get("models_called"),
    })

# Wait for LangSmith traces / Aguarda os traces do LangSmith
wait_for_all_tracers()

print("FIXED_ACTION_METRICS_RESULTS_OK")

In [ ]:
import pandas as pd

# Create readable labels for scenario names / Cria rótulos legíveis para os nomes dos cenários
case_labels = {
    "commercial_lead": "Entrada comercial",
    "technical_request": "Integração técnica",
    "complex_analysis": "Diagnóstico operacional",
    "general_question": "Explicação conceitual",
}

# Create readable labels for detected intents / Cria rótulos legíveis para as intenções detectadas
intent_labels = {
    "commercial_lead": "Lead comercial",
    "technical_request": "Solicitação técnica",
    "complex_analysis": "Análise complexa",
    "general_question": "Pergunta geral",
    "support_request": "Suporte",
    "unclear": "Indefinido",
}

# Create readable labels for priority values / Cria rótulos legíveis para os valores de prioridade
priority_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

# Create readable labels for confidence values / Cria rótulos legíveis para os valores de confiança
confidence_labels = {
    "low": "Baixa",
    "medium": "Média",
    "high": "Alta",
}

# Create readable labels for LLM routes / Cria rótulos legíveis para as rotas dos LLMs
route_labels = {
    "fast": "Rápido",
    "agent": "Agente",
    "reasoning": "Raciocínio",
    "general": "Geral",
}

# Create dataframe from normalized results / Cria DataFrame a partir dos resultados normalizados
metrics_df = pd.DataFrame(normalized_case_results).copy()

# Apply readable labels / Aplica rótulos legíveis
metrics_df["case"] = metrics_df["case"].map(case_labels).fillna(metrics_df["case"])
metrics_df["intent"] = metrics_df["intent"].map(intent_labels).fillna(metrics_df["intent"])
metrics_df["priority"] = metrics_df["priority"].map(priority_labels).fillna(metrics_df["priority"])
metrics_df["confidence"] = metrics_df["confidence"].map(confidence_labels).fillna(metrics_df["confidence"])
metrics_df["selected_route"] = metrics_df["selected_route"].map(route_labels).fillna(metrics_df["selected_route"])

# Rename columns for final monitoring table / Renomeia colunas para a tabela final de monitoramento
metrics_df = metrics_df.rename(
    columns={
        "case": "Cenário de Teste",
        "intent": "Intenção Detectada",
        "priority": "Prioridade",
        "confidence": "Confiança",
        "selected_route": "Rota LLM",
        "model_used": "Modelo Executor",
        "requires_human_review": "Revisão Humana",
        "recommended_action": "Ação Recomendada",
        "llm_calls": "Chamadas LLM",
        "input_tokens": "Tokens Entrada",
        "output_tokens": "Tokens Saída",
        "total_tokens": "Tokens Totais",
        "avg_tokens_per_call": "Média Tokens/Chamada",
        "models_called": "Modelos Chamados",
    }
)

# Reorder columns / Reordena as colunas
metrics_df = metrics_df[
    [
        "Cenário de Teste",
        "Intenção Detectada",
        "Prioridade",
        "Confiança",
        "Rota LLM",
        "Modelo Executor",
        "Modelos Chamados",
        "Chamadas LLM",
        "Tokens Entrada",
        "Tokens Saída",
        "Tokens Totais",
        "Média Tokens/Chamada",
        "Revisão Humana",
        "Ação Recomendada",
    ]
]

# Display styled metrics table / Exibe tabela de métricas estilizada
metrics_styled_table = (
    metrics_df.style
    .hide(axis="index")
    .set_properties(
        **{
            "border": "1px solid #4b5563",
            "padding": "10px",
            "vertical-align": "middle",
            "font-size": "13px",
        }
    )
    .set_properties(
        subset=[
            "Cenário de Teste",
            "Intenção Detectada",
            "Prioridade",
            "Confiança",
            "Rota LLM",
            "Modelo Executor",
            "Modelos Chamados",
            "Chamadas LLM",
            "Tokens Entrada",
            "Tokens Saída",
            "Tokens Totais",
            "Média Tokens/Chamada",
            "Revisão Humana",
        ],
        **{
            "text-align": "center",
            "white-space": "normal",
            "vertical-align": "middle",
        }
    )
    .set_properties(
        subset=["Ação Recomendada"],
        **{
            "text-align": "left",
            "white-space": "normal",
            "vertical-align": "middle",
            "min-width": "340px",
            "max-width": "480px",
        }
    )
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("padding", "12px"),
                    ("text-align", "center"),
                    ("vertical-align", "middle"),
                    ("font-weight", "bold"),
                    ("background-color", "#111827"),
                    ("color", "#f9fafb"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #4b5563"),
                    ("border-bottom", "2px solid #374151"),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("width", "100%"),
                ],
            },
        ]
    )
)

metrics_styled_table

In [ ]:
import os
import pandas as pd
from datetime import timezone
from dotenv import load_dotenv
from langsmith import Client

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data: dict | None, path: list[str], default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default
        current = current.get(key)

    return current if current is not None else default


# Convert value to readable text / Converte valor para texto legível
def to_readable_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        return ", ".join(str(item) for item in value)
    if isinstance(value, dict):
        return str(value)
    return str(value)


# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}

    model_candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "model"]),
    ]

    for candidate in model_candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Calculate latency in seconds / Calcula latência em segundos
def calculate_latency_seconds(run) -> float | None:
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Convert datetime to local-readable text / Converte data/hora para texto legível
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return "Erro"

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return "Erro"

    return "Sucesso"


# Get error message / Obtém mensagem de erro
def get_error_message(run) -> str:
    error = getattr(run, "error", None)

    if not error:
        return ""

    return str(error)


# Fetch LangSmith runs / Busca runs no LangSmith
def fetch_langsmith_runs(project_name: str, limit: int = 100) -> list:
    try:
        return list(
            langsmith_client.list_runs(
                project_name=project_name,
                limit=limit,
            )
        )
    except Exception as error:
        raise RuntimeError(f"Erro ao buscar runs do LangSmith: {error}")


# Build LangSmith monitoring table / Cria tabela de monitoramento do LangSmith
def build_langsmith_monitoring_table(project_name: str, limit: int = 100) -> pd.DataFrame:
    runs = fetch_langsmith_runs(project_name=project_name, limit=limit)

    rows = []

    for run in runs:
        tokens = extract_tokens_from_run(run)

        run_type = getattr(run, "run_type", "")
        run_name = getattr(run, "name", "")
        run_id = getattr(run, "id", "")
        trace_id = getattr(run, "trace_id", "")
        parent_run_id = getattr(run, "parent_run_id", None)
        tags = getattr(run, "tags", []) or []
        latency = calculate_latency_seconds(run)
        status = determine_run_status(run)
        error_message = get_error_message(run)

        rows.append(
            {
                "Projeto": project_name,
                "Run Name": run_name,
                "Run Type": run_type,
                "Status": status,
                "Erro": error_message,
                "Modelo": extract_model_name_from_run(run),
                "Tokens Entrada": tokens["input_tokens"],
                "Tokens Saída": tokens["output_tokens"],
                "Tokens Totais": tokens["total_tokens"],
                "Latência (s)": latency,
                "Tags": to_readable_text(tags),
                "Run ID": str(run_id),
                "Trace ID": str(trace_id),
                "Parent Run ID": str(parent_run_id) if parent_run_id else "",
                "Início": format_datetime(getattr(run, "start_time", None)),
                "Fim": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Generate LangSmith monitoring dataframe / Gera DataFrame de monitoramento do LangSmith
langsmith_metrics_df = build_langsmith_monitoring_table(
    project_name=LANGSMITH_PROJECT_NAME,
    limit=100,
)

print("LANGSMITH_MONITORING_TABLE_CREATED")
print("Project / Projeto:", LANGSMITH_PROJECT_NAME)
print("Runs loaded / Runs carregados:", len(langsmith_metrics_df))

langsmith_metrics_df.head(20)

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from langsmith import Client

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define expected project and Groq models / Define o projeto e os modelos Groq esperados
project_display_name = "sentrya-ops-v2-agentic-ai-desk"

expected_llms = {
    "llama-3.1-8b-instant": {
        "role": "Classificação rápida",
        "purpose": "Validação, pré-processamento e roteamento inicial",
    },
    "openai/gpt-oss-20b": {
        "role": "Agente principal",
        "purpose": "Orquestração, decisão operacional, JSON e tool calling",
    },
    "openai/gpt-oss-120b": {
        "role": "Raciocínio pesado",
        "purpose": "Análise complexa, priorização e decisões críticas",
    },
    "llama-3.3-70b-versatile": {
        "role": "Comunicação geral",
        "purpose": "Síntese, explicação natural e resposta final",
    },
}


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data, path, default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

    return current if current is not None else default


# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    serialized = getattr(run, "serialized", None) or {}

    candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "ls_model_name"]),
        safe_nested_get(extra, ["metadata", "model_name"]),
        safe_nested_get(serialized, ["kwargs", "model"]),
        safe_nested_get(serialized, ["kwargs", "model_name"]),
    ]

    for candidate in candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Calculate run latency in seconds / Calcula a latência do run em segundos
def calculate_latency_seconds(run):
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return "Erro"

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return "Erro"

    return "Sucesso"


# Format datetime / Formata data e hora
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Fetch all LangSmith runs and normalize into dataframe / Busca todos os runs do LangSmith e normaliza em DataFrame
def fetch_langsmith_runs_dataframe(project_name: str, limit: int = 500) -> pd.DataFrame:
    runs = list(
        langsmith_client.list_runs(
            project_name=project_name,
            limit=limit,
        )
    )

    rows = []

    for run in runs:
        tokens = extract_tokens_from_run(run)

        rows.append(
            {
                "project": project_name,
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "model_name": extract_model_name_from_run(run),
                "status": determine_run_status(run),
                "error": str(getattr(run, "error", "") or ""),
                "input_tokens": tokens["input_tokens"],
                "output_tokens": tokens["output_tokens"],
                "total_tokens": tokens["total_tokens"],
                "latency_seconds": calculate_latency_seconds(run),
                "tags": ", ".join(getattr(run, "tags", []) or []),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Build consolidated project + LLM monitoring table / Cria tabela consolidada de projeto + LLMs
def build_project_and_llm_monitoring_table(project_name: str, limit: int = 500) -> pd.DataFrame:
    all_runs_df = fetch_langsmith_runs_dataframe(project_name=project_name, limit=limit)

    if all_runs_df.empty:
        return pd.DataFrame(
            [
                {
                    "Camada": "Projeto",
                    "Nome": project_display_name,
                    "Função": "Projeto completo",
                    "Uso Principal": "Visão consolidada do Sentrya Ops V2",
                    "Runs LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            ]
        )

    llm_runs_df = all_runs_df[all_runs_df["run_type"] == "llm"].copy()

    total_project_runs = len(all_runs_df)
    total_llm_calls = len(llm_runs_df)

    project_success_count = int((all_runs_df["status"] == "Sucesso").sum())
    project_error_count = int((all_runs_df["status"] == "Erro").sum())
    project_error_rate = round((project_error_count / total_project_runs) * 100, 2) if total_project_runs else 0

    project_input_tokens = int(llm_runs_df["input_tokens"].sum()) if not llm_runs_df.empty else 0
    project_output_tokens = int(llm_runs_df["output_tokens"].sum()) if not llm_runs_df.empty else 0
    project_total_tokens = int(llm_runs_df["total_tokens"].sum()) if not llm_runs_df.empty else 0
    project_avg_tokens_per_call = round(project_total_tokens / total_llm_calls, 2) if total_llm_calls else 0

    project_avg_latency = (
        round(all_runs_df["latency_seconds"].dropna().mean(), 3)
        if all_runs_df["latency_seconds"].notna().any()
        else 0
    )

    project_last_execution = all_runs_df["start_time"].max() if "start_time" in all_runs_df else ""

    if project_error_count == 0:
        project_status = "Saudável"
    elif project_success_count > 0:
        project_status = "Atenção"
    else:
        project_status = "Com erro"

    summary_rows = [
        {
            "Camada": "Projeto",
            "Nome": project_display_name,
            "Função": "Sentrya Ops V2 completo",
            "Uso Principal": "Visão consolidada de todos os runs e consumo total dos LLMs",
            "Runs LangSmith": total_project_runs,
            "Chamadas LLM": total_llm_calls,
            "Sucessos": project_success_count,
            "Erros": project_error_count,
            "Taxa de Erro": f"{project_error_rate}%",
            "Tokens Entrada": project_input_tokens,
            "Tokens Saída": project_output_tokens,
            "Tokens Totais": project_total_tokens,
            "Média Tokens/Chamada": project_avg_tokens_per_call,
            "Latência Média (s)": project_avg_latency,
            "Última Execução": project_last_execution,
            "Status Geral": project_status,
        }
    ]

    for model_name, model_info in expected_llms.items():
        if llm_runs_df.empty:
            model_runs = pd.DataFrame()
        else:
            model_runs = llm_runs_df[llm_runs_df["model_name"] == model_name]

        if model_runs.empty:
            summary_rows.append(
                {
                    "Camada": "LLM",
                    "Nome": model_name,
                    "Função": model_info["role"],
                    "Uso Principal": model_info["purpose"],
                    "Runs LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            )
            continue

        model_total_calls = len(model_runs)
        model_success_count = int((model_runs["status"] == "Sucesso").sum())
        model_error_count = int((model_runs["status"] == "Erro").sum())
        model_error_rate = round((model_error_count / model_total_calls) * 100, 2) if model_total_calls else 0

        model_input_tokens = int(model_runs["input_tokens"].sum())
        model_output_tokens = int(model_runs["output_tokens"].sum())
        model_total_tokens = int(model_runs["total_tokens"].sum())
        model_avg_tokens_per_call = round(model_total_tokens / model_total_calls, 2) if model_total_calls else 0

        model_avg_latency = (
            round(model_runs["latency_seconds"].dropna().mean(), 3)
            if model_runs["latency_seconds"].notna().any()
            else 0
        )

        model_last_execution = model_runs["start_time"].max() if "start_time" in model_runs else ""

        if model_error_count == 0:
            model_status = "Saudável"
        elif model_success_count > 0:
            model_status = "Atenção"
        else:
            model_status = "Com erro"

        summary_rows.append(
            {
                "Camada": "LLM",
                "Nome": model_name,
                "Função": model_info["role"],
                "Uso Principal": model_info["purpose"],
                "Runs LangSmith": model_total_calls,
                "Chamadas LLM": model_total_calls,
                "Sucessos": model_success_count,
                "Erros": model_error_count,
                "Taxa de Erro": f"{model_error_rate}%",
                "Tokens Entrada": model_input_tokens,
                "Tokens Saída": model_output_tokens,
                "Tokens Totais": model_total_tokens,
                "Média Tokens/Chamada": model_avg_tokens_per_call,
                "Latência Média (s)": model_avg_latency,
                "Última Execução": model_last_execution,
                "Status Geral": model_status,
            }
        )

    return pd.DataFrame(summary_rows)


# Generate consolidated monitoring dataframe / Gera DataFrame consolidado de monitoramento
project_llm_monitoring_df = build_project_and_llm_monitoring_table(
    project_name=LANGSMITH_PROJECT_NAME,
    limit=500,
)

print("PROJECT_AND_LLM_MONITORING_TABLE_OK")
print("Projeto LangSmith:", LANGSMITH_PROJECT_NAME)
print("Linhas da tabela:", len(project_llm_monitoring_df))

project_llm_monitoring_df

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from langsmith import Client

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")

# Define LangSmith API maximum run limit / Define o limite máximo aceito pela API do LangSmith
MAX_LANGSMITH_RUNS = 100

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define expected project and Groq models / Define o projeto e os modelos Groq esperados
project_display_name = "sentrya-ops-v2-agentic-ai-desk"

expected_llms = {
    "llama-3.1-8b-instant": {
        "role": "Classificação rápida",
        "purpose": "Validação, pré-processamento e roteamento inicial",
    },
    "openai/gpt-oss-20b": {
        "role": "Agente principal",
        "purpose": "Orquestração, decisão operacional, JSON e tool calling",
    },
    "openai/gpt-oss-120b": {
        "role": "Raciocínio pesado",
        "purpose": "Análise complexa, priorização e decisões críticas",
    },
    "llama-3.3-70b-versatile": {
        "role": "Comunicação geral",
        "purpose": "Síntese, explicação natural e resposta final",
    },
}


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data, path, default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

    return current if current is not None else default


# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    serialized = getattr(run, "serialized", None) or {}

    candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "ls_model_name"]),
        safe_nested_get(extra, ["metadata", "model_name"]),
        safe_nested_get(serialized, ["kwargs", "model"]),
        safe_nested_get(serialized, ["kwargs", "model_name"]),
    ]

    for candidate in candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Calculate run latency in seconds / Calcula a latência do run em segundos
def calculate_latency_seconds(run):
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return "Erro"

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return "Erro"

    return "Sucesso"


# Format datetime / Formata data e hora
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Fetch all LangSmith runs and normalize into dataframe / Busca todos os runs do LangSmith e normaliza em DataFrame
def fetch_langsmith_runs_dataframe(project_name: str, limit: int = MAX_LANGSMITH_RUNS) -> pd.DataFrame:
    safe_limit = min(limit, MAX_LANGSMITH_RUNS)

    runs = list(
        langsmith_client.list_runs(
            project_name=project_name,
            limit=safe_limit,
        )
    )

    rows = []

    for run in runs:
        tokens = extract_tokens_from_run(run)

        rows.append(
            {
                "project": project_name,
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "model_name": extract_model_name_from_run(run),
                "status": determine_run_status(run),
                "error": str(getattr(run, "error", "") or ""),
                "input_tokens": tokens["input_tokens"],
                "output_tokens": tokens["output_tokens"],
                "total_tokens": tokens["total_tokens"],
                "latency_seconds": calculate_latency_seconds(run),
                "tags": ", ".join(getattr(run, "tags", []) or []),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Build consolidated project + LLM monitoring table / Cria tabela consolidada de projeto + LLMs
def build_project_and_llm_monitoring_table(project_name: str, limit: int = MAX_LANGSMITH_RUNS) -> pd.DataFrame:
    all_runs_df = fetch_langsmith_runs_dataframe(project_name=project_name, limit=limit)

    if all_runs_df.empty:
        return pd.DataFrame(
            [
                {
                    "Camada": "Projeto",
                    "Nome": project_display_name,
                    "Função": "Projeto completo",
                    "Uso Principal": "Visão consolidada do Sentrya Ops V2",
                    "Runs LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            ]
        )

    llm_runs_df = all_runs_df[all_runs_df["run_type"] == "llm"].copy()

    total_project_runs = len(all_runs_df)
    total_llm_calls = len(llm_runs_df)

    project_success_count = int((all_runs_df["status"] == "Sucesso").sum())
    project_error_count = int((all_runs_df["status"] == "Erro").sum())
    project_error_rate = round((project_error_count / total_project_runs) * 100, 2) if total_project_runs else 0

    project_input_tokens = int(llm_runs_df["input_tokens"].sum()) if not llm_runs_df.empty else 0
    project_output_tokens = int(llm_runs_df["output_tokens"].sum()) if not llm_runs_df.empty else 0
    project_total_tokens = int(llm_runs_df["total_tokens"].sum()) if not llm_runs_df.empty else 0
    project_avg_tokens_per_call = round(project_total_tokens / total_llm_calls, 2) if total_llm_calls else 0

    project_avg_latency = (
        round(all_runs_df["latency_seconds"].dropna().mean(), 3)
        if all_runs_df["latency_seconds"].notna().any()
        else 0
    )

    project_last_execution = all_runs_df["start_time"].max() if "start_time" in all_runs_df else ""

    if project_error_count == 0:
        project_status = "Saudável"
    elif project_success_count > 0:
        project_status = "Atenção"
    else:
        project_status = "Com erro"

    summary_rows = [
        {
            "Camada": "Projeto",
            "Nome": project_display_name,
            "Função": "Sentrya Ops V2 completo",
            "Uso Principal": "Visão consolidada de todos os runs e consumo total dos LLMs",
            "Runs LangSmith": total_project_runs,
            "Chamadas LLM": total_llm_calls,
            "Sucessos": project_success_count,
            "Erros": project_error_count,
            "Taxa de Erro": f"{project_error_rate}%",
            "Tokens Entrada": project_input_tokens,
            "Tokens Saída": project_output_tokens,
            "Tokens Totais": project_total_tokens,
            "Média Tokens/Chamada": project_avg_tokens_per_call,
            "Latência Média (s)": project_avg_latency,
            "Última Execução": project_last_execution,
            "Status Geral": project_status,
        }
    ]

    for model_name, model_info in expected_llms.items():
        if llm_runs_df.empty:
            model_runs = pd.DataFrame()
        else:
            model_runs = llm_runs_df[llm_runs_df["model_name"] == model_name]

        if model_runs.empty:
            summary_rows.append(
                {
                    "Camada": "LLM",
                    "Nome": model_name,
                    "Função": model_info["role"],
                    "Uso Principal": model_info["purpose"],
                    "Runs LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            )
            continue

        model_total_calls = len(model_runs)
        model_success_count = int((model_runs["status"] == "Sucesso").sum())
        model_error_count = int((model_runs["status"] == "Erro").sum())
        model_error_rate = round((model_error_count / model_total_calls) * 100, 2) if model_total_calls else 0

        model_input_tokens = int(model_runs["input_tokens"].sum())
        model_output_tokens = int(model_runs["output_tokens"].sum())
        model_total_tokens = int(model_runs["total_tokens"].sum())
        model_avg_tokens_per_call = round(model_total_tokens / model_total_calls, 2) if model_total_calls else 0

        model_avg_latency = (
            round(model_runs["latency_seconds"].dropna().mean(), 3)
            if model_runs["latency_seconds"].notna().any()
            else 0
        )

        model_last_execution = model_runs["start_time"].max() if "start_time" in model_runs else ""

        if model_error_count == 0:
            model_status = "Saudável"
        elif model_success_count > 0:
            model_status = "Atenção"
        else:
            model_status = "Com erro"

        summary_rows.append(
            {
                "Camada": "LLM",
                "Nome": model_name,
                "Função": model_info["role"],
                "Uso Principal": model_info["purpose"],
                "Runs LangSmith": model_total_calls,
                "Chamadas LLM": model_total_calls,
                "Sucessos": model_success_count,
                "Erros": model_error_count,
                "Taxa de Erro": f"{model_error_rate}%",
                "Tokens Entrada": model_input_tokens,
                "Tokens Saída": model_output_tokens,
                "Tokens Totais": model_total_tokens,
                "Média Tokens/Chamada": model_avg_tokens_per_call,
                "Latência Média (s)": model_avg_latency,
                "Última Execução": model_last_execution,
                "Status Geral": model_status,
            }
        )

    return pd.DataFrame(summary_rows)


# Generate consolidated monitoring dataframe / Gera DataFrame consolidado de monitoramento
project_llm_monitoring_df = build_project_and_llm_monitoring_table(
    project_name=LANGSMITH_PROJECT_NAME,
    limit=MAX_LANGSMITH_RUNS,
)

print("PROJECT_AND_LLM_MONITORING_TABLE_OK")
print("Projeto LangSmith:", LANGSMITH_PROJECT_NAME)
print("Limite usado:", MAX_LANGSMITH_RUNS)
print("Linhas da tabela:", len(project_llm_monitoring_df))

project_llm_monitoring_df

In [ ]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from langsmith import Client
from IPython.display import display, clear_output

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define expected project and Groq models / Define o projeto e os modelos Groq esperados
project_display_name = "sentrya-ops-v2-agentic-ai-desk"

expected_llms = {
    "llama-3.1-8b-instant": {
        "role": "Classificação rápida",
        "purpose": "Validação, pré-processamento e roteamento inicial",
    },
    "openai/gpt-oss-20b": {
        "role": "Agente principal",
        "purpose": "Orquestração, decisão operacional, JSON e tool calling",
    },
    "openai/gpt-oss-120b": {
        "role": "Raciocínio pesado",
        "purpose": "Análise complexa, priorização e decisões críticas",
    },
    "llama-3.3-70b-versatile": {
        "role": "Comunicação geral",
        "purpose": "Síntese, explicação natural e resposta final",
    },
}


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data, path, default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

    return current if current is not None else default


# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    serialized = getattr(run, "serialized", None) or {}

    candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "ls_model_name"]),
        safe_nested_get(extra, ["metadata", "model_name"]),
        safe_nested_get(serialized, ["kwargs", "model"]),
        safe_nested_get(serialized, ["kwargs", "model_name"]),
    ]

    for candidate in candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Calculate run latency in seconds / Calcula a latência do run em segundos
def calculate_latency_seconds(run):
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return "Erro"

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return "Erro"

    return "Sucesso"


# Format datetime / Formata data e hora
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Fetch all LangSmith runs without manual 100 cap / Busca todos os runs do LangSmith sem limitar manualmente em 100
def fetch_all_langsmith_runs_dataframe(project_name: str) -> pd.DataFrame:
    runs = list(
        langsmith_client.list_runs(
            project_name=project_name
        )
    )

    rows = []

    for run in runs:
        tokens = extract_tokens_from_run(run)

        rows.append(
            {
                "project": project_name,
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "model_name": extract_model_name_from_run(run),
                "status": determine_run_status(run),
                "error": str(getattr(run, "error", "") or ""),
                "input_tokens": tokens["input_tokens"],
                "output_tokens": tokens["output_tokens"],
                "total_tokens": tokens["total_tokens"],
                "latency_seconds": calculate_latency_seconds(run),
                "tags": ", ".join(getattr(run, "tags", []) or []),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Build consolidated project + 4 LLMs monitoring table / Cria tabela consolidada do projeto + 4 LLMs
def build_project_and_llm_monitoring_table(project_name: str) -> pd.DataFrame:
    all_runs_df = fetch_all_langsmith_runs_dataframe(project_name=project_name)

    if all_runs_df.empty:
        return pd.DataFrame(
            [
                {
                    "Camada": "Projeto",
                    "Nome": project_display_name,
                    "Função": "Sentrya Ops V2 completo",
                    "Uso Principal": "Visão consolidada do projeto",
                    "Runs LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            ]
        )

    llm_runs_df = all_runs_df[all_runs_df["run_type"] == "llm"].copy()

    total_project_runs = len(all_runs_df)
    total_llm_calls = len(llm_runs_df)

    project_success_count = int((all_runs_df["status"] == "Sucesso").sum())
    project_error_count = int((all_runs_df["status"] == "Erro").sum())
    project_error_rate = round((project_error_count / total_project_runs) * 100, 2) if total_project_runs else 0

    # Token totals are calculated from LLM runs only to avoid duplicated parent-chain tokens /
    # Totais de tokens são calculados apenas por runs LLM para evitar duplicação dos runs pais
    project_input_tokens = int(llm_runs_df["input_tokens"].sum()) if not llm_runs_df.empty else 0
    project_output_tokens = int(llm_runs_df["output_tokens"].sum()) if not llm_runs_df.empty else 0
    project_total_tokens = int(llm_runs_df["total_tokens"].sum()) if not llm_runs_df.empty else 0
    project_avg_tokens_per_call = round(project_total_tokens / total_llm_calls, 2) if total_llm_calls else 0

    project_avg_latency = (
        round(all_runs_df["latency_seconds"].dropna().mean(), 3)
        if all_runs_df["latency_seconds"].notna().any()
        else 0
    )

    project_last_execution = all_runs_df["start_time"].max() if "start_time" in all_runs_df else ""

    if project_error_count == 0:
        project_status = "Saudável"
    elif project_success_count > 0:
        project_status = "Atenção"
    else:
        project_status = "Com erro"

    summary_rows = [
        {
            "Camada": "Projeto",
            "Nome": project_display_name,
            "Função": "Sentrya Ops V2 completo",
            "Uso Principal": "Visão consolidada de todos os runs e consumo total dos LLMs",
            "Runs LangSmith": total_project_runs,
            "Chamadas LLM": total_llm_calls,
            "Sucessos": project_success_count,
            "Erros": project_error_count,
            "Taxa de Erro": f"{project_error_rate}%",
            "Tokens Entrada": project_input_tokens,
            "Tokens Saída": project_output_tokens,
            "Tokens Totais": project_total_tokens,
            "Média Tokens/Chamada": project_avg_tokens_per_call,
            "Latência Média (s)": project_avg_latency,
            "Última Execução": project_last_execution,
            "Status Geral": project_status,
        }
    ]

    for model_name, model_info in expected_llms.items():
        model_runs = llm_runs_df[llm_runs_df["model_name"] == model_name] if not llm_runs_df.empty else pd.DataFrame()

        if model_runs.empty:
            summary_rows.append(
                {
                    "Camada": "LLM",
                    "Nome": model_name,
                    "Função": model_info["role"],
                    "Uso Principal": model_info["purpose"],
                    "Runs LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            )
            continue

        total_calls = len(model_runs)
        success_count = int((model_runs["status"] == "Sucesso").sum())
        error_count = int((model_runs["status"] == "Erro").sum())
        error_rate = round((error_count / total_calls) * 100, 2) if total_calls else 0

        input_tokens = int(model_runs["input_tokens"].sum())
        output_tokens = int(model_runs["output_tokens"].sum())
        total_tokens = int(model_runs["total_tokens"].sum())
        avg_tokens_per_call = round(total_tokens / total_calls, 2) if total_calls else 0

        avg_latency = (
            round(model_runs["latency_seconds"].dropna().mean(), 3)
            if model_runs["latency_seconds"].notna().any()
            else 0
        )

        last_execution = model_runs["start_time"].max() if "start_time" in model_runs else ""

        if error_count == 0:
            status_general = "Saudável"
        elif success_count > 0:
            status_general = "Atenção"
        else:
            status_general = "Com erro"

        summary_rows.append(
            {
                "Camada": "LLM",
                "Nome": model_name,
                "Função": model_info["role"],
                "Uso Principal": model_info["purpose"],
                "Runs LangSmith": total_calls,
                "Chamadas LLM": total_calls,
                "Sucessos": success_count,
                "Erros": error_count,
                "Taxa de Erro": f"{error_rate}%",
                "Tokens Entrada": input_tokens,
                "Tokens Saída": output_tokens,
                "Tokens Totais": total_tokens,
                "Média Tokens/Chamada": avg_tokens_per_call,
                "Latência Média (s)": avg_latency,
                "Última Execução": last_execution,
                "Status Geral": status_general,
            }
        )

    return pd.DataFrame(summary_rows)


# Style monitoring table / Estiliza tabela de monitoramento
def style_project_llm_monitoring_table(df: pd.DataFrame):
    styled_table = (
        df.style
        .hide(axis="index")
        .set_properties(
            **{
                "border": "1px solid #4b5563",
                "padding": "10px",
                "vertical-align": "middle",
                "font-size": "13px",
            }
        )
        .set_properties(
            subset=[
                "Camada",
                "Nome",
                "Função",
                "Runs LangSmith",
                "Chamadas LLM",
                "Sucessos",
                "Erros",
                "Taxa de Erro",
                "Tokens Entrada",
                "Tokens Saída",
                "Tokens Totais",
                "Média Tokens/Chamada",
                "Latência Média (s)",
                "Última Execução",
                "Status Geral",
            ],
            **{
                "text-align": "center",
                "white-space": "normal",
                "vertical-align": "middle",
            }
        )
        .set_properties(
            subset=["Uso Principal"],
            **{
                "text-align": "left",
                "white-space": "normal",
                "vertical-align": "middle",
                "min-width": "300px",
                "max-width": "460px",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("padding", "12px"),
                        ("text-align", "center"),
                        ("vertical-align", "middle"),
                        ("font-weight", "bold"),
                        ("background-color", "#111827"),
                        ("color", "#f9fafb"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("border-bottom", "2px solid #374151"),
                    ],
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                    ],
                },
            ]
        )
    )

    return styled_table


# Refresh monitoring table once / Atualiza a tabela de monitoramento uma vez
def refresh_project_llm_monitoring_table():
    monitoring_df = build_project_and_llm_monitoring_table(
        project_name=LANGSMITH_PROJECT_NAME
    )

    print("PROJECT_AND_LLM_MONITORING_TABLE_OK")
    print("Projeto LangSmith:", LANGSMITH_PROJECT_NAME)
    print("Runs LangSmith encontrados:", int(monitoring_df.loc[0, "Runs LangSmith"]))
    print("Linhas da tabela:", len(monitoring_df))

    display(style_project_llm_monitoring_table(monitoring_df))

    return monitoring_df


# Run one refresh / Executa uma atualização
project_llm_monitoring_df = refresh_project_llm_monitoring_table()

In [ ]:
# Auto-refresh monitoring table / Atualiza automaticamente a tabela de monitoramento
for cycle in range(5):
    clear_output(wait=True)

    print(f"Atualização {cycle + 1}/5")
    project_llm_monitoring_df = refresh_project_llm_monitoring_table()

    time.sleep(30)

In [ ]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langsmith import Client

# Detect project root if needed / Detecta a raiz do projeto se necessário
try:
    PROJECT_ROOT
except NameError:
    CURRENT_DIR = Path.cwd()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define project and expected Groq models / Define o projeto e os modelos Groq esperados
PROJECT_DISPLAY_NAME = "sentrya-ops-v2-agentic-ai-desk"

EXPECTED_LLMS = {
    "llama-3.1-8b-instant": {
        "role": "Classificação rápida",
        "purpose": "Validação, pré-processamento e roteamento inicial",
    },
    "openai/gpt-oss-20b": {
        "role": "Agente principal",
        "purpose": "Orquestração, decisão operacional, JSON e tool calling",
    },
    "openai/gpt-oss-120b": {
        "role": "Raciocínio pesado",
        "purpose": "Análise complexa, priorização e decisões críticas",
    },
    "llama-3.3-70b-versatile": {
        "role": "Comunicação geral",
        "purpose": "Síntese, explicação natural e resposta final",
    },
}


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data, path, default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default
        current = current.get(key)

    return current if current is not None else default


# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    serialized = getattr(run, "serialized", None) or {}

    candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "ls_model_name"]),
        safe_nested_get(extra, ["metadata", "model_name"]),
        safe_nested_get(serialized, ["kwargs", "model"]),
        safe_nested_get(serialized, ["kwargs", "model_name"]),
    ]

    for candidate in candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Calculate run latency in seconds / Calcula a latência do run em segundos
def calculate_latency_seconds(run):
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return "Erro"

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return "Erro"

    return "Sucesso"


# Format datetime / Formata data e hora
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Fetch LangSmith runs and normalize / Busca runs do LangSmith e normaliza
def fetch_langsmith_runs_dataframe(project_name: str) -> pd.DataFrame:
    runs = list(langsmith_client.list_runs(project_name=project_name))

    rows = []

    for run in runs:
        tokens = extract_tokens_from_run(run)

        run_id = str(getattr(run, "id", "") or "")
        trace_id = str(getattr(run, "trace_id", "") or run_id)
        parent_run_id = getattr(run, "parent_run_id", None)

        rows.append(
            {
                "project": project_name,
                "run_id": run_id,
                "trace_id": trace_id,
                "parent_run_id": str(parent_run_id) if parent_run_id else "",
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "model_name": extract_model_name_from_run(run),
                "status": determine_run_status(run),
                "error": str(getattr(run, "error", "") or ""),
                "input_tokens": tokens["input_tokens"],
                "output_tokens": tokens["output_tokens"],
                "total_tokens": tokens["total_tokens"],
                "latency_seconds": calculate_latency_seconds(run),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Build project and LLM monitoring table with correct trace count / Cria tabela projeto + LLM com contagem correta de traces
def build_project_and_llm_monitoring_table(project_name: str) -> pd.DataFrame:
    all_runs_df = fetch_langsmith_runs_dataframe(project_name)

    if all_runs_df.empty:
        return pd.DataFrame()

    llm_runs_df = all_runs_df[all_runs_df["run_type"] == "llm"].copy()

    # Count unique traces, matching LangSmith project-level trace count / Conta traces únicos, alinhado ao contador do projeto no LangSmith
    total_traces = all_runs_df["trace_id"].nunique()

    # Count trace-level errors: a trace is considered errored if any run in that trace has error / Conta erros por trace
    trace_status_df = (
        all_runs_df
        .groupby("trace_id")
        .agg(
            trace_has_error=("status", lambda values: any(value == "Erro" for value in values)),
            latest_start=("start_time", "max"),
        )
        .reset_index()
    )

    error_traces = int(trace_status_df["trace_has_error"].sum())
    success_traces = int(total_traces - error_traces)
    project_error_rate = round((error_traces / total_traces) * 100, 2) if total_traces else 0

    total_llm_calls = len(llm_runs_df)
    project_input_tokens = int(llm_runs_df["input_tokens"].sum()) if not llm_runs_df.empty else 0
    project_output_tokens = int(llm_runs_df["output_tokens"].sum()) if not llm_runs_df.empty else 0
    project_total_tokens = int(llm_runs_df["total_tokens"].sum()) if not llm_runs_df.empty else 0
    project_avg_tokens = round(project_total_tokens / total_llm_calls, 2) if total_llm_calls else 0

    project_avg_latency = (
        round(all_runs_df["latency_seconds"].dropna().mean(), 3)
        if all_runs_df["latency_seconds"].notna().any()
        else 0
    )

    project_last_execution = all_runs_df["start_time"].max()

    if error_traces == 0:
        project_status = "Saudável"
    elif success_traces > 0:
        project_status = "Atenção"
    else:
        project_status = "Com erro"

    rows = [
        {
            "Camada": "Projeto",
            "Nome": PROJECT_DISPLAY_NAME,
            "Função": "Sentrya Ops V2 completo",
            "Uso Principal": "Visão consolidada do projeto, traces e consumo total dos LLMs",
            "Traces LangSmith": total_traces,
            "Chamadas LLM": total_llm_calls,
            "Sucessos": success_traces,
            "Erros": error_traces,
            "Taxa de Erro": f"{project_error_rate}%",
            "Tokens Entrada": project_input_tokens,
            "Tokens Saída": project_output_tokens,
            "Tokens Totais": project_total_tokens,
            "Média Tokens/Chamada": project_avg_tokens,
            "Latência Média (s)": project_avg_latency,
            "Última Execução": project_last_execution,
            "Status Geral": project_status,
        }
    ]

    for model_name, model_info in EXPECTED_LLMS.items():
        model_runs = llm_runs_df[llm_runs_df["model_name"] == model_name] if not llm_runs_df.empty else pd.DataFrame()

        if model_runs.empty:
            rows.append(
                {
                    "Camada": "LLM",
                    "Nome": model_name,
                    "Função": model_info["role"],
                    "Uso Principal": model_info["purpose"],
                    "Traces LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            )
            continue

        model_traces = model_runs["trace_id"].nunique()
        model_calls = len(model_runs)
        model_errors = int((model_runs["status"] == "Erro").sum())
        model_successes = int((model_runs["status"] == "Sucesso").sum())
        model_error_rate = round((model_errors / model_calls) * 100, 2) if model_calls else 0

        model_input_tokens = int(model_runs["input_tokens"].sum())
        model_output_tokens = int(model_runs["output_tokens"].sum())
        model_total_tokens = int(model_runs["total_tokens"].sum())
        model_avg_tokens = round(model_total_tokens / model_calls, 2) if model_calls else 0

        model_avg_latency = (
            round(model_runs["latency_seconds"].dropna().mean(), 3)
            if model_runs["latency_seconds"].notna().any()
            else 0
        )

        model_last_execution = model_runs["start_time"].max()

        if model_errors == 0:
            model_status = "Saudável"
        elif model_successes > 0:
            model_status = "Atenção"
        else:
            model_status = "Com erro"

        rows.append(
            {
                "Camada": "LLM",
                "Nome": model_name,
                "Função": model_info["role"],
                "Uso Principal": model_info["purpose"],
                "Traces LangSmith": model_traces,
                "Chamadas LLM": model_calls,
                "Sucessos": model_successes,
                "Erros": model_errors,
                "Taxa de Erro": f"{model_error_rate}%",
                "Tokens Entrada": model_input_tokens,
                "Tokens Saída": model_output_tokens,
                "Tokens Totais": model_total_tokens,
                "Média Tokens/Chamada": model_avg_tokens,
                "Latência Média (s)": model_avg_latency,
                "Última Execução": model_last_execution,
                "Status Geral": model_status,
            }
        )

    return pd.DataFrame(rows)


# Style monitoring table / Estiliza a tabela de monitoramento
def style_project_llm_monitoring_table(df: pd.DataFrame):
    return (
        df.style
        .hide(axis="index")
        .set_properties(
            **{
                "border": "1px solid #4b5563",
                "padding": "10px",
                "vertical-align": "middle",
                "font-size": "13px",
            }
        )
        .set_properties(
            subset=[
                "Camada",
                "Nome",
                "Função",
                "Traces LangSmith",
                "Chamadas LLM",
                "Sucessos",
                "Erros",
                "Taxa de Erro",
                "Tokens Entrada",
                "Tokens Saída",
                "Tokens Totais",
                "Média Tokens/Chamada",
                "Latência Média (s)",
                "Última Execução",
                "Status Geral",
            ],
            **{
                "text-align": "center",
                "white-space": "normal",
                "vertical-align": "middle",
            }
        )
        .set_properties(
            subset=["Uso Principal"],
            **{
                "text-align": "left",
                "white-space": "normal",
                "vertical-align": "middle",
                "min-width": "300px",
                "max-width": "460px",
            }
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("padding", "12px"),
                        ("text-align", "center"),
                        ("vertical-align", "middle"),
                        ("font-weight", "bold"),
                        ("background-color", "#111827"),
                        ("color", "#f9fafb"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("border-bottom", "2px solid #374151"),
                    ],
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                    ],
                },
            ]
        )
    )


# Generate corrected monitoring table / Gera a tabela de monitoramento corrigida
project_llm_monitoring_df = build_project_and_llm_monitoring_table(LANGSMITH_PROJECT_NAME)

print("PROJECT_AND_LLM_MONITORING_TABLE_CORRECTED_OK")
print("Projeto LangSmith:", LANGSMITH_PROJECT_NAME)
print("Traces únicos encontrados:", int(project_llm_monitoring_df.loc[0, "Traces LangSmith"]))
print("Linhas da tabela:", len(project_llm_monitoring_df))

style_project_llm_monitoring_table(project_llm_monitoring_df)v

In [ ]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langsmith import Client

# Detect project root if needed / Detecta a raiz do projeto se necessário
try:
    PROJECT_ROOT
except NameError:
    CURRENT_DIR = Path.cwd()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define project display name / Define o nome exibido do projeto
PROJECT_DISPLAY_NAME = "sentrya-ops-v2-agentic-ai-desk"

# Define expected Groq models and roles / Define os modelos Groq esperados e suas funções
EXPECTED_LLMS = {
    "llama-3.1-8b-instant": {
        "role": "Classificação rápida",
        "purpose": "Validação, pré-processamento e roteamento inicial",
    },
    "openai/gpt-oss-20b": {
        "role": "Agente principal",
        "purpose": "Orquestração, decisão operacional, JSON e tool calling",
    },
    "openai/gpt-oss-120b": {
        "role": "Raciocínio pesado",
        "purpose": "Análise complexa, priorização e decisões críticas",
    },
    "llama-3.3-70b-versatile": {
        "role": "Comunicação geral",
        "purpose": "Síntese, explicação natural e resposta final",
    },
}


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data, path, default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

    return current if current is not None else default


# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    serialized = getattr(run, "serialized", None) or {}

    candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "ls_model_name"]),
        safe_nested_get(extra, ["metadata", "model_name"]),
        safe_nested_get(serialized, ["kwargs", "model"]),
        safe_nested_get(serialized, ["kwargs", "model_name"]),
    ]

    for candidate in candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Calculate run latency in seconds / Calcula a latência do run em segundos
def calculate_latency_seconds(run):
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return "Erro"

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return "Erro"

    return "Sucesso"


# Format datetime / Formata data e hora
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Fetch LangSmith runs and normalize into dataframe / Busca runs do LangSmith e normaliza em DataFrame
def fetch_langsmith_runs_dataframe(project_name: str) -> pd.DataFrame:
    runs = list(langsmith_client.list_runs(project_name=project_name))
    rows = []

    for run in runs:
        tokens = extract_tokens_from_run(run)

        run_id = str(getattr(run, "id", "") or "")
        trace_id = str(getattr(run, "trace_id", "") or run_id)
        parent_run_id = getattr(run, "parent_run_id", None)

        rows.append(
            {
                "project": project_name,
                "run_id": run_id,
                "trace_id": trace_id,
                "parent_run_id": str(parent_run_id) if parent_run_id else "",
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "model_name": extract_model_name_from_run(run),
                "status": determine_run_status(run),
                "error": str(getattr(run, "error", "") or ""),
                "input_tokens": tokens["input_tokens"],
                "output_tokens": tokens["output_tokens"],
                "total_tokens": tokens["total_tokens"],
                "latency_seconds": calculate_latency_seconds(run),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Build project and LLM monitoring table / Cria tabela de monitoramento do projeto e dos LLMs
def build_project_and_llm_monitoring_table(project_name: str) -> pd.DataFrame:
    all_runs_df = fetch_langsmith_runs_dataframe(project_name)

    if all_runs_df.empty:
        return pd.DataFrame(
            [
                {
                    "Camada": "Projeto",
                    "Nome": PROJECT_DISPLAY_NAME,
                    "Função": "Sentrya Ops V2 completo",
                    "Uso Principal": "Visão consolidada do projeto, traces e consumo dos LLMs",
                    "Traces LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            ]
        )

    llm_runs_df = all_runs_df[all_runs_df["run_type"] == "llm"].copy()

    # Count unique traces to match LangSmith project tracing counter / Conta traces únicos para alinhar com o contador de tracing do LangSmith
    total_traces = all_runs_df["trace_id"].nunique()

    # Count trace-level errors / Conta erros no nível do trace
    trace_status_df = (
        all_runs_df
        .groupby("trace_id")
        .agg(
            trace_has_error=("status", lambda values: any(value == "Erro" for value in values)),
            latest_start=("start_time", "max"),
        )
        .reset_index()
    )

    error_traces = int(trace_status_df["trace_has_error"].sum())
    success_traces = int(total_traces - error_traces)
    project_error_rate = round((error_traces / total_traces) * 100, 2) if total_traces else 0

    total_llm_calls = len(llm_runs_df)

    # Token totals are calculated only from LLM runs to avoid duplicate parent-chain tokens /
    # Totais de tokens são calculados apenas nos runs LLM para evitar duplicação de tokens dos runs pais
    project_input_tokens = int(llm_runs_df["input_tokens"].sum()) if not llm_runs_df.empty else 0
    project_output_tokens = int(llm_runs_df["output_tokens"].sum()) if not llm_runs_df.empty else 0
    project_total_tokens = int(llm_runs_df["total_tokens"].sum()) if not llm_runs_df.empty else 0
    project_avg_tokens = round(project_total_tokens / total_llm_calls, 2) if total_llm_calls else 0

    project_avg_latency = (
        round(all_runs_df["latency_seconds"].dropna().mean(), 3)
        if all_runs_df["latency_seconds"].notna().any()
        else 0
    )

    project_last_execution = all_runs_df["start_time"].max()

    if error_traces == 0:
        project_status = "Saudável"
    elif success_traces > 0:
        project_status = "Atenção"
    else:
        project_status = "Com erro"

    rows = [
        {
            "Camada": "Projeto",
            "Nome": PROJECT_DISPLAY_NAME,
            "Função": "Sentrya Ops V2 completo",
            "Uso Principal": "Visão consolidada do projeto, traces e consumo total dos LLMs",
            "Traces LangSmith": total_traces,
            "Chamadas LLM": total_llm_calls,
            "Sucessos": success_traces,
            "Erros": error_traces,
            "Taxa de Erro": f"{project_error_rate}%",
            "Tokens Entrada": project_input_tokens,
            "Tokens Saída": project_output_tokens,
            "Tokens Totais": project_total_tokens,
            "Média Tokens/Chamada": project_avg_tokens,
            "Latência Média (s)": project_avg_latency,
            "Última Execução": project_last_execution,
            "Status Geral": project_status,
        }
    ]

    for model_name, model_info in EXPECTED_LLMS.items():
        if llm_runs_df.empty:
            model_runs = pd.DataFrame()
        else:
            model_runs = llm_runs_df[llm_runs_df["model_name"] == model_name]

        if model_runs.empty:
            rows.append(
                {
                    "Camada": "LLM",
                    "Nome": model_name,
                    "Função": model_info["role"],
                    "Uso Principal": model_info["purpose"],
                    "Traces LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                }
            )
            continue

        model_traces = model_runs["trace_id"].nunique()
        model_calls = len(model_runs)
        model_errors = int((model_runs["status"] == "Erro").sum())
        model_successes = int((model_runs["status"] == "Sucesso").sum())
        model_error_rate = round((model_errors / model_calls) * 100, 2) if model_calls else 0

        model_input_tokens = int(model_runs["input_tokens"].sum())
        model_output_tokens = int(model_runs["output_tokens"].sum())
        model_total_tokens = int(model_runs["total_tokens"].sum())
        model_avg_tokens = round(model_total_tokens / model_calls, 2) if model_calls else 0

        model_avg_latency = (
            round(model_runs["latency_seconds"].dropna().mean(), 3)
            if model_runs["latency_seconds"].notna().any()
            else 0
        )

        model_last_execution = model_runs["start_time"].max()

        if model_errors == 0:
            model_status = "Saudável"
        elif model_successes > 0:
            model_status = "Atenção"
        else:
            model_status = "Com erro"

        rows.append(
            {
                "Camada": "LLM",
                "Nome": model_name,
                "Função": model_info["role"],
                "Uso Principal": model_info["purpose"],
                "Traces LangSmith": model_traces,
                "Chamadas LLM": model_calls,
                "Sucessos": model_successes,
                "Erros": model_errors,
                "Taxa de Erro": f"{model_error_rate}%",
                "Tokens Entrada": model_input_tokens,
                "Tokens Saída": model_output_tokens,
                "Tokens Totais": model_total_tokens,
                "Média Tokens/Chamada": model_avg_tokens,
                "Latência Média (s)": model_avg_latency,
                "Última Execução": model_last_execution,
                "Status Geral": model_status,
            }
        )

    return pd.DataFrame(rows)


# Style monitoring table / Estiliza a tabela de monitoramento
def style_project_llm_monitoring_table(df: pd.DataFrame):
    styled_table = (
        df.style
        .hide(axis="index")
        .set_properties(
            **{
                "border": "1px solid #4b5563",
                "padding": "10px",
                "vertical-align": "middle",
                "font-size": "13px",
            }
        )
        .set_properties(
            subset=[
                "Camada",
                "Nome",
                "Função",
                "Traces LangSmith",
                "Chamadas LLM",
                "Sucessos",
                "Erros",
                "Taxa de Erro",
                "Tokens Entrada",
                "Tokens Saída",
                "Tokens Totais",
                "Média Tokens/Chamada",
                "Latência Média (s)",
                "Última Execução",
                "Status Geral",
            ],
            **{
                "text-align": "center",
                "white-space": "normal",
                "vertical-align": "middle",
            },
        )
        .set_properties(
            subset=["Uso Principal"],
            **{
                "text-align": "left",
                "white-space": "normal",
                "vertical-align": "middle",
                "min-width": "300px",
                "max-width": "460px",
            },
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("padding", "12px"),
                        ("text-align", "center"),
                        ("vertical-align", "middle"),
                        ("font-weight", "bold"),
                        ("background-color", "#111827"),
                        ("color", "#f9fafb"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("border-bottom", "2px solid #374151"),
                    ],
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                    ],
                },
            ]
        )
    )

    return styled_table


# Generate corrected monitoring table / Gera a tabela corrigida de monitoramento
project_llm_monitoring_df = build_project_and_llm_monitoring_table(LANGSMITH_PROJECT_NAME)

print("PROJECT_AND_LLM_MONITORING_TABLE_CORRECTED_OK")
print("Projeto LangSmith:", LANGSMITH_PROJECT_NAME)
print("Traces únicos encontrados:", int(project_llm_monitoring_df.loc[0, "Traces LangSmith"]))
print("Linhas da tabela:", len(project_llm_monitoring_df))

style_project_llm_monitoring_table(project_llm_monitoring_df)

In [ ]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langsmith import Client

# Detect project root if needed / Detecta a raiz do projeto se necessário
try:
    PROJECT_ROOT
except NameError:
    CURRENT_DIR = Path.cwd()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define display project name / Define o nome exibido do projeto
PROJECT_DISPLAY_NAME = "sentrya-ops-v2-agentic-ai-desk"

# Define expected Groq models and their agent roles / Define os modelos Groq esperados e suas funções no agente
EXPECTED_LLMS = {
    "llama-3.1-8b-instant": {
        "role": "Classificação rápida",
        "purpose": "Validação, pré-processamento e roteamento inicial",
    },
    "openai/gpt-oss-20b": {
        "role": "Agente principal",
        "purpose": "Orquestração, decisão operacional, JSON estruturado e tool calling",
    },
    "openai/gpt-oss-120b": {
        "role": "Raciocínio pesado",
        "purpose": "Análise complexa, priorização e decisões críticas",
    },
    "llama-3.3-70b-versatile": {
        "role": "Comunicação geral",
        "purpose": "Síntese, explicação natural e resposta final",
    },
}


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data, path, default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default

        current = current.get(key)

    return current if current is not None else default


# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    serialized = getattr(run, "serialized", None) or {}

    candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "ls_model_name"]),
        safe_nested_get(extra, ["metadata", "model_name"]),
        safe_nested_get(serialized, ["kwargs", "model"]),
        safe_nested_get(serialized, ["kwargs", "model_name"]),
    ]

    for candidate in candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai o uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Calculate run latency in seconds / Calcula a latência do run em segundos
def calculate_latency_seconds(run):
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Determine if a run has error / Determina se um run possui erro
def has_run_error(run) -> bool:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return True

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return True

    return False


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    return "Erro" if has_run_error(run) else "Sucesso"


# Format datetime / Formata data e hora
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Fetch root traces from LangSmith / Busca traces raiz do LangSmith
def fetch_root_traces_dataframe(project_name: str) -> pd.DataFrame:
    try:
        root_runs = list(
            langsmith_client.list_runs(
                project_name=project_name,
                is_root=True,
            )
        )
    except TypeError:
        all_runs = list(
            langsmith_client.list_runs(
                project_name=project_name,
            )
        )

        root_runs = [
            run for run in all_runs
            if getattr(run, "parent_run_id", None) is None
        ]

    rows = []

    for run in root_runs:
        run_id = str(getattr(run, "id", "") or "")
        trace_id = str(getattr(run, "trace_id", "") or run_id)

        rows.append(
            {
                "project": project_name,
                "run_id": run_id,
                "trace_id": trace_id,
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "status": determine_run_status(run),
                "has_error": has_run_error(run),
                "error": str(getattr(run, "error", "") or ""),
                "latency_seconds": calculate_latency_seconds(run),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Fetch LLM runs from LangSmith / Busca runs LLM do LangSmith
def fetch_llm_runs_dataframe(project_name: str) -> pd.DataFrame:
    try:
        llm_runs = list(
            langsmith_client.list_runs(
                project_name=project_name,
                run_type="llm",
            )
        )
    except TypeError:
        all_runs = list(
            langsmith_client.list_runs(
                project_name=project_name,
            )
        )

        llm_runs = [
            run for run in all_runs
            if str(getattr(run, "run_type", "") or "").lower() == "llm"
        ]

    rows = []

    for run in llm_runs:
        tokens = extract_tokens_from_run(run)

        run_id = str(getattr(run, "id", "") or "")
        trace_id = str(getattr(run, "trace_id", "") or run_id)

        rows.append(
            {
                "project": project_name,
                "run_id": run_id,
                "trace_id": trace_id,
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "model_name": extract_model_name_from_run(run),
                "status": determine_run_status(run),
                "has_error": has_run_error(run),
                "error": str(getattr(run, "error", "") or ""),
                "input_tokens": tokens["input_tokens"],
                "output_tokens": tokens["output_tokens"],
                "total_tokens": tokens["total_tokens"],
                "latency_seconds": calculate_latency_seconds(run),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Build LangSmith truth table with project + 4 LLMs / Cria tabela verdadeira do LangSmith com projeto + 4 LLMs
def build_langsmith_project_llm_table(project_name: str) -> pd.DataFrame:
    root_traces_df = fetch_root_traces_dataframe(project_name)
    llm_runs_df = fetch_llm_runs_dataframe(project_name)

    if root_traces_df.empty:
        project_total_calls = 0
        project_errors = 0
        project_successes = 0
        project_error_rate = 0
        project_avg_latency = 0
        project_last_execution = "Sem execução"
        project_status = "Sem execução"
    else:
        # Project-level numbers are based on root traces to match LangSmith UI / Números do projeto usam traces raiz para alinhar com a UI do LangSmith
        project_total_calls = len(root_traces_df)
        project_errors = int(root_traces_df["has_error"].sum())
        project_successes = int(project_total_calls - project_errors)
        project_error_rate = round((project_errors / project_total_calls) * 100, 2) if project_total_calls else 0

        project_avg_latency = (
            round(root_traces_df["latency_seconds"].dropna().mean(), 3)
            if root_traces_df["latency_seconds"].notna().any()
            else 0
        )

        project_last_execution = root_traces_df["start_time"].max()

        if project_errors == 0:
            project_status = "Saudável"
        elif project_successes > 0:
            project_status = "Atenção"
        else:
            project_status = "Com erro"

    # Token totals are based only on LLM runs to avoid duplicated parent-run usage / Tokens usam apenas runs LLM para evitar duplicação dos runs pais
    if llm_runs_df.empty:
        project_llm_calls = 0
        project_input_tokens = 0
        project_output_tokens = 0
        project_total_tokens = 0
        project_avg_tokens = 0
    else:
        project_llm_calls = len(llm_runs_df)
        project_input_tokens = int(llm_runs_df["input_tokens"].sum())
        project_output_tokens = int(llm_runs_df["output_tokens"].sum())
        project_total_tokens = int(llm_runs_df["total_tokens"].sum())
        project_avg_tokens = round(project_total_tokens / project_llm_calls, 2) if project_llm_calls else 0

    rows = [
        {
            "Camada": "Projeto",
            "Nome": PROJECT_DISPLAY_NAME,
            "Função": "Sentrya Ops V2 completo",
            "Uso Principal": "Visão consolidada do projeto com dados reais do LangSmith",
            "Chamadas/Traces LangSmith": project_total_calls,
            "Chamadas LLM": project_llm_calls,
            "Sucessos": project_successes,
            "Erros": project_errors,
            "Taxa de Erro": f"{project_error_rate}%",
            "Tokens Entrada": project_input_tokens,
            "Tokens Saída": project_output_tokens,
            "Tokens Totais": project_total_tokens,
            "Média Tokens/Chamada": project_avg_tokens,
            "Latência Média (s)": project_avg_latency,
            "Última Execução": project_last_execution,
            "Status Geral": project_status,
            "Fonte": "LangSmith root traces + LLM runs",
        }
    ]

    for model_name, model_info in EXPECTED_LLMS.items():
        if llm_runs_df.empty:
            model_runs = pd.DataFrame()
        else:
            model_runs = llm_runs_df[llm_runs_df["model_name"] == model_name]

        if model_runs.empty:
            rows.append(
                {
                    "Camada": "LLM",
                    "Nome": model_name,
                    "Função": model_info["role"],
                    "Uso Principal": model_info["purpose"],
                    "Chamadas/Traces LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                    "Fonte": "LangSmith LLM runs",
                }
            )
            continue

        model_total_calls = len(model_runs)
        model_errors = int(model_runs["has_error"].sum())
        model_successes = int(model_total_calls - model_errors)
        model_error_rate = round((model_errors / model_total_calls) * 100, 2) if model_total_calls else 0

        model_input_tokens = int(model_runs["input_tokens"].sum())
        model_output_tokens = int(model_runs["output_tokens"].sum())
        model_total_tokens = int(model_runs["total_tokens"].sum())
        model_avg_tokens = round(model_total_tokens / model_total_calls, 2) if model_total_calls else 0

        model_avg_latency = (
            round(model_runs["latency_seconds"].dropna().mean(), 3)
            if model_runs["latency_seconds"].notna().any()
            else 0
        )

        model_last_execution = model_runs["start_time"].max()

        if model_errors == 0:
            model_status = "Saudável"
        elif model_successes > 0:
            model_status = "Atenção"
        else:
            model_status = "Com erro"

        rows.append(
            {
                "Camada": "LLM",
                "Nome": model_name,
                "Função": model_info["role"],
                "Uso Principal": model_info["purpose"],
                "Chamadas/Traces LangSmith": model_runs["trace_id"].nunique(),
                "Chamadas LLM": model_total_calls,
                "Sucessos": model_successes,
                "Erros": model_errors,
                "Taxa de Erro": f"{model_error_rate}%",
                "Tokens Entrada": model_input_tokens,
                "Tokens Saída": model_output_tokens,
                "Tokens Totais": model_total_tokens,
                "Média Tokens/Chamada": model_avg_tokens,
                "Latência Média (s)": model_avg_latency,
                "Última Execução": model_last_execution,
                "Status Geral": model_status,
                "Fonte": "LangSmith LLM runs",
            }
        )

    return pd.DataFrame(rows)


# Style LangSmith truth table / Estiliza a tabela verdadeira do LangSmith
def style_langsmith_truth_table(df: pd.DataFrame):
    return (
        df.style
        .hide(axis="index")
        .set_properties(
            **{
                "border": "1px solid #4b5563",
                "padding": "10px",
                "vertical-align": "middle",
                "font-size": "13px",
            }
        )
        .set_properties(
            subset=[
                "Camada",
                "Nome",
                "Função",
                "Chamadas/Traces LangSmith",
                "Chamadas LLM",
                "Sucessos",
                "Erros",
                "Taxa de Erro",
                "Tokens Entrada",
                "Tokens Saída",
                "Tokens Totais",
                "Média Tokens/Chamada",
                "Latência Média (s)",
                "Última Execução",
                "Status Geral",
                "Fonte",
            ],
            **{
                "text-align": "center",
                "white-space": "normal",
                "vertical-align": "middle",
            },
        )
        .set_properties(
            subset=["Uso Principal"],
            **{
                "text-align": "left",
                "white-space": "normal",
                "vertical-align": "middle",
                "min-width": "320px",
                "max-width": "480px",
            },
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("padding", "12px"),
                        ("text-align", "center"),
                        ("vertical-align", "middle"),
                        ("font-weight", "bold"),
                        ("background-color", "#111827"),
                        ("color", "#f9fafb"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("border-bottom", "2px solid #374151"),
                    ],
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                    ],
                },
            ]
        )
    )


# Generate final LangSmith truth table / Gera a tabela final baseada no LangSmith
langsmith_truth_df = build_langsmith_project_llm_table(LANGSMITH_PROJECT_NAME)

print("LANGSMITH_PROJECT_AND_LLM_TRUTH_TABLE_OK")
print("Projeto LangSmith:", LANGSMITH_PROJECT_NAME)
print("Chamadas/Traces LangSmith:", int(langsmith_truth_df.loc[0, "Chamadas/Traces LangSmith"]))
print("Sucessos:", int(langsmith_truth_df.loc[0, "Sucessos"]))
print("Erros:", int(langsmith_truth_df.loc[0, "Erros"]))
print("Linhas da tabela:", len(langsmith_truth_df))

style_langsmith_truth_table(langsmith_truth_df)

In [ ]:
import os
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from langsmith import Client
from IPython.display import display, clear_output


# =========================
# CONFIGURATION / CONFIGURAÇÃO
# =========================

# Detect project root if needed / Detecta a raiz do projeto se necessário
try:
    PROJECT_ROOT
except NameError:
    CURRENT_DIR = Path.cwd()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

# Load environment variables / Carrega as variáveis de ambiente
env_path = PROJECT_ROOT / ".env"
load_dotenv(env_path, override=True)

# Define LangSmith project / Define o projeto LangSmith
LANGSMITH_PROJECT_NAME = os.getenv("LANGSMITH_PROJECT", "sentrya-ops-v2-agentic-ai-desk")
PROJECT_DISPLAY_NAME = "sentrya-ops-v2-agentic-ai-desk"

# Auto-refresh settings / Configurações de atualização automática
REFRESH_SECONDS = 30
AUTO_REFRESH_CYCLES = 5

# Create LangSmith client / Cria o cliente do LangSmith
langsmith_client = Client(
    api_key=os.getenv("LANGSMITH_API_KEY"),
    api_url=os.getenv("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com"),
)

# Define expected Groq models and their roles / Define os modelos Groq esperados e suas funções
EXPECTED_LLMS = {
    "llama-3.1-8b-instant": {
        "role": "Classificação rápida",
        "purpose": "Validação, pré-processamento e roteamento inicial",
    },
    "openai/gpt-oss-20b": {
        "role": "Agente principal",
        "purpose": "Orquestração, decisão operacional, JSON estruturado e tool calling",
    },
    "openai/gpt-oss-120b": {
        "role": "Raciocínio pesado",
        "purpose": "Análise complexa, priorização e decisões críticas",
    },
    "llama-3.3-70b-versatile": {
        "role": "Comunicação geral",
        "purpose": "Síntese, explicação natural e resposta final",
    },
}


# =========================
# SAFE HELPERS / FUNÇÕES SEGURAS
# =========================

# Convert object or Pydantic model to dictionary / Converte objeto ou modelo Pydantic para dicionário
def to_dict_safe(value):
    if value is None:
        return {}

    if isinstance(value, dict):
        return value

    if hasattr(value, "model_dump"):
        try:
            return value.model_dump()
        except Exception:
            pass

    if hasattr(value, "dict"):
        try:
            return value.dict()
        except Exception:
            pass

    output = {}

    for attr in dir(value):
        if attr.startswith("_"):
            continue

        try:
            attr_value = getattr(value, attr)
        except Exception:
            continue

        if callable(attr_value):
            continue

        output[attr] = attr_value

    return output


# Safely get nested dictionary values / Busca valores aninhados com segurança
def safe_nested_get(data, path, default=None):
    current = data or {}

    for key in path:
        if not isinstance(current, dict):
            return default
        current = current.get(key)

    return current if current is not None else default


# Recursively find first value by candidate keys / Procura recursivamente o primeiro valor por chaves candidatas
def recursive_find_key(data, candidate_keys):
    if data is None:
        return None

    if isinstance(data, dict):
        for key, value in data.items():
            if key in candidate_keys and value is not None:
                return value

        for value in data.values():
            found = recursive_find_key(value, candidate_keys)
            if found is not None:
                return found

    if isinstance(data, list):
        for item in data:
            found = recursive_find_key(item, candidate_keys)
            if found is not None:
                return found

    return None


# Convert value to int safely / Converte valor para inteiro com segurança
def to_int_safe(value, default=0):
    try:
        if value is None:
            return default

        if isinstance(value, str):
            value = value.replace(",", "").replace(".", "")

        return int(float(value))
    except Exception:
        return default


# Convert value to float safely / Converte valor para float com segurança
def to_float_safe(value, default=0.0):
    try:
        if value is None:
            return default

        if isinstance(value, str):
            value = value.replace("%", "").replace(",", ".")

        return float(value)
    except Exception:
        return default


# Format datetime / Formata data e hora
def format_datetime(value) -> str:
    if value is None:
        return ""

    try:
        return value.astimezone().strftime("%Y-%m-%d %H:%M:%S")
    except Exception:
        return str(value)


# Calculate run latency in seconds / Calcula a latência do run em segundos
def calculate_latency_seconds(run):
    start_time = getattr(run, "start_time", None)
    end_time = getattr(run, "end_time", None)

    if not start_time or not end_time:
        return None

    return round((end_time - start_time).total_seconds(), 3)


# Determine if a run has error / Determina se um run possui erro
def has_run_error(run) -> bool:
    error = getattr(run, "error", None)
    status = getattr(run, "status", None)

    if error:
        return True

    if status and str(status).lower() in ["error", "failed", "failure"]:
        return True

    return False


# Determine run status / Determina o status do run
def determine_run_status(run) -> str:
    return "Erro" if has_run_error(run) else "Sucesso"


# =========================
# PROJECT DASHBOARD METRICS / MÉTRICAS DO PROJETO
# =========================

# Read LangSmith project object / Lê o objeto do projeto no LangSmith
def read_langsmith_project(project_name: str):
    attempts = [
        lambda: langsmith_client.read_project(project_name=project_name),
        lambda: langsmith_client.read_project(project_name),
    ]

    for attempt in attempts:
        try:
            project = attempt()
            return project, to_dict_safe(project)
        except Exception:
            pass

    # Fallback through list_projects / Alternativa usando list_projects
    try:
        for project in langsmith_client.list_projects():
            project_dict = to_dict_safe(project)
            names = [
                getattr(project, "name", None),
                project_dict.get("name"),
                project_dict.get("session_name"),
            ]

            if project_name in names:
                return project, project_dict

            if project_name in str(project_dict):
                return project, project_dict

    except Exception:
        pass

    return None, {}


# Extract exact dashboard metrics from project object / Extrai métricas exatas do objeto do projeto
def get_project_dashboard_metrics(project_name: str) -> dict:
    project_obj, project_dict = read_langsmith_project(project_name)

    # Top-level fields first / Campos de primeiro nível primeiro
    count_candidates = [
        "run_count",
        "trace_count",
        "total_run_count",
        "total_trace_count",
        "total_runs",
        "total_traces",
        "num_runs",
        "num_traces",
        "count",
    ]

    token_candidates = [
        "total_tokens",
        "token_count",
        "tokens",
        "total_token_count",
    ]

    error_rate_candidates = [
        "error_rate",
        "feedback_error_rate",
    ]

    latency_p50_candidates = [
        "latency_p50",
        "p50_latency",
        "latency50",
        "latency_p50_ms",
    ]

    latency_p99_candidates = [
        "latency_p99",
        "p99_latency",
        "latency99",
        "latency_p99_ms",
    ]

    dashboard_count = None
    dashboard_tokens = None
    dashboard_error_rate = None
    dashboard_latency_p50 = None
    dashboard_latency_p99 = None

    for key in count_candidates:
        if key in project_dict and project_dict.get(key) is not None:
            dashboard_count = project_dict.get(key)
            break

    for key in token_candidates:
        if key in project_dict and project_dict.get(key) is not None:
            dashboard_tokens = project_dict.get(key)
            break

    for key in error_rate_candidates:
        if key in project_dict and project_dict.get(key) is not None:
            dashboard_error_rate = project_dict.get(key)
            break

    for key in latency_p50_candidates:
        if key in project_dict and project_dict.get(key) is not None:
            dashboard_latency_p50 = project_dict.get(key)
            break

    for key in latency_p99_candidates:
        if key in project_dict and project_dict.get(key) is not None:
            dashboard_latency_p99 = project_dict.get(key)
            break

    # Recursive fallback / Alternativa recursiva
    if dashboard_count is None:
        dashboard_count = recursive_find_key(project_dict, set(count_candidates))

    if dashboard_tokens is None:
        dashboard_tokens = recursive_find_key(project_dict, set(token_candidates))

    if dashboard_error_rate is None:
        dashboard_error_rate = recursive_find_key(project_dict, set(error_rate_candidates))

    if dashboard_latency_p50 is None:
        dashboard_latency_p50 = recursive_find_key(project_dict, set(latency_p50_candidates))

    if dashboard_latency_p99 is None:
        dashboard_latency_p99 = recursive_find_key(project_dict, set(latency_p99_candidates))

    return {
        "project_found": project_obj is not None,
        "project_raw": project_dict,
        "dashboard_count": to_int_safe(dashboard_count, default=None),
        "dashboard_tokens": to_int_safe(dashboard_tokens, default=None),
        "dashboard_error_rate_raw": dashboard_error_rate,
        "dashboard_latency_p50_raw": dashboard_latency_p50,
        "dashboard_latency_p99_raw": dashboard_latency_p99,
    }


# =========================
# RUN EXTRACTION / EXTRAÇÃO DE RUNS
# =========================

# Extract model name from LangSmith run / Extrai o nome do modelo do run do LangSmith
def extract_model_name_from_run(run) -> str:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    serialized = getattr(run, "serialized", None) or {}

    candidates = [
        getattr(run, "model_name", None),
        metadata.get("ls_model_name"),
        metadata.get("model_name"),
        metadata.get("model"),
        safe_nested_get(extra, ["invocation_params", "model"]),
        safe_nested_get(extra, ["invocation_params", "model_name"]),
        safe_nested_get(extra, ["metadata", "ls_model_name"]),
        safe_nested_get(extra, ["metadata", "model_name"]),
        safe_nested_get(serialized, ["kwargs", "model"]),
        safe_nested_get(serialized, ["kwargs", "model_name"]),
    ]

    for candidate in candidates:
        if candidate:
            return str(candidate)

    return "Não identificado"


# Extract token usage from LangSmith run / Extrai o uso de tokens do run do LangSmith
def extract_tokens_from_run(run) -> dict:
    extra = getattr(run, "extra", None) or {}
    metadata = extra.get("metadata", {}) if isinstance(extra, dict) else {}
    outputs = getattr(run, "outputs", None) or {}

    input_tokens = (
        getattr(run, "prompt_tokens", None)
        or getattr(run, "input_tokens", None)
        or metadata.get("input_tokens")
        or metadata.get("prompt_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "input_tokens"])
        or safe_nested_get(metadata, ["token_usage", "prompt_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "input_tokens"])
        or safe_nested_get(outputs, ["token_usage", "prompt_tokens"])
        or 0
    )

    output_tokens = (
        getattr(run, "completion_tokens", None)
        or getattr(run, "output_tokens", None)
        or metadata.get("output_tokens")
        or metadata.get("completion_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "output_tokens"])
        or safe_nested_get(metadata, ["token_usage", "completion_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "output_tokens"])
        or safe_nested_get(outputs, ["token_usage", "completion_tokens"])
        or 0
    )

    total_tokens = (
        getattr(run, "total_tokens", None)
        or metadata.get("total_tokens")
        or safe_nested_get(metadata, ["usage_metadata", "total_tokens"])
        or safe_nested_get(metadata, ["token_usage", "total_tokens"])
        or safe_nested_get(outputs, ["usage_metadata", "total_tokens"])
        or safe_nested_get(outputs, ["token_usage", "total_tokens"])
        or int(input_tokens or 0) + int(output_tokens or 0)
    )

    return {
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or 0),
    }


# Fetch root traces for error count / Busca traces raiz para contar erros
def fetch_root_traces_dataframe(project_name: str) -> pd.DataFrame:
    try:
        root_runs = list(
            langsmith_client.list_runs(
                project_name=project_name,
                is_root=True,
            )
        )
    except TypeError:
        all_runs = list(langsmith_client.list_runs(project_name=project_name))
        root_runs = [
            run for run in all_runs
            if getattr(run, "parent_run_id", None) is None
        ]

    rows = []

    for run in root_runs:
        run_id = str(getattr(run, "id", "") or "")
        trace_id = str(getattr(run, "trace_id", "") or run_id)

        rows.append(
            {
                "run_id": run_id,
                "trace_id": trace_id,
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "status": determine_run_status(run),
                "has_error": has_run_error(run),
                "error": str(getattr(run, "error", "") or ""),
                "latency_seconds": calculate_latency_seconds(run),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# Fetch LLM runs for model metrics / Busca runs LLM para métricas por modelo
def fetch_llm_runs_dataframe(project_name: str) -> pd.DataFrame:
    try:
        llm_runs = list(
            langsmith_client.list_runs(
                project_name=project_name,
                run_type="llm",
            )
        )
    except TypeError:
        all_runs = list(langsmith_client.list_runs(project_name=project_name))
        llm_runs = [
            run for run in all_runs
            if str(getattr(run, "run_type", "") or "").lower() == "llm"
        ]

    rows = []

    for run in llm_runs:
        tokens = extract_tokens_from_run(run)

        run_id = str(getattr(run, "id", "") or "")
        trace_id = str(getattr(run, "trace_id", "") or run_id)

        rows.append(
            {
                "run_id": run_id,
                "trace_id": trace_id,
                "run_name": getattr(run, "name", ""),
                "run_type": str(getattr(run, "run_type", "") or "").lower(),
                "model_name": extract_model_name_from_run(run),
                "status": determine_run_status(run),
                "has_error": has_run_error(run),
                "error": str(getattr(run, "error", "") or ""),
                "input_tokens": tokens["input_tokens"],
                "output_tokens": tokens["output_tokens"],
                "total_tokens": tokens["total_tokens"],
                "latency_seconds": calculate_latency_seconds(run),
                "start_time": format_datetime(getattr(run, "start_time", None)),
                "end_time": format_datetime(getattr(run, "end_time", None)),
            }
        )

    return pd.DataFrame(rows)


# =========================
# TABLE BUILDING / CRIAÇÃO DA TABELA
# =========================

# Build LangSmith real-time project + LLM table / Cria tabela em tempo real do projeto + LLMs
def build_langsmith_realtime_project_llm_table(project_name: str) -> tuple[pd.DataFrame, dict]:
    dashboard_metrics = get_project_dashboard_metrics(project_name)
    root_traces_df = fetch_root_traces_dataframe(project_name)
    llm_runs_df = fetch_llm_runs_dataframe(project_name)

    dashboard_count = dashboard_metrics["dashboard_count"]
    dashboard_tokens = dashboard_metrics["dashboard_tokens"]

    root_total = len(root_traces_df) if not root_traces_df.empty else 0
    root_errors = int(root_traces_df["has_error"].sum()) if not root_traces_df.empty else 0

    root_avg_latency = (
        round(root_traces_df["latency_seconds"].dropna().mean(), 3)
        if not root_traces_df.empty and root_traces_df["latency_seconds"].notna().any()
        else 0
    )

    root_last_execution = (
        root_traces_df["start_time"].max()
        if not root_traces_df.empty
        else "Sem execução"
    )

    # The project total must come from LangSmith project dashboard count /
    # O total do projeto deve vir do contador do projeto no LangSmith
    project_total = int(dashboard_count) if dashboard_count is not None else int(root_total)

    # Error count uses root traces because the project object may expose only rate /
    # Erros usam traces raiz porque o objeto do projeto pode expor apenas taxa
    project_errors = int(root_errors)

    # Success count is total dashboard count minus root trace errors /
    # Sucessos = total da dashboard menos erros dos traces raiz
    project_successes = max(project_total - project_errors, 0)

    project_error_rate = round((project_errors / project_total) * 100, 2) if project_total else 0

    if llm_runs_df.empty:
        project_llm_calls = 0
        project_input_tokens = 0
        project_output_tokens = 0
        project_total_tokens = dashboard_tokens if dashboard_tokens is not None else 0
        project_avg_tokens = 0
    else:
        project_llm_calls = len(llm_runs_df)
        project_input_tokens = int(llm_runs_df["input_tokens"].sum())
        project_output_tokens = int(llm_runs_df["output_tokens"].sum())

        # Prefer dashboard total tokens for project row when available /
        # Preferir tokens totais da dashboard na linha do projeto quando disponível
        project_total_tokens = int(dashboard_tokens) if dashboard_tokens is not None else int(llm_runs_df["total_tokens"].sum())
        project_avg_tokens = round(project_total_tokens / project_llm_calls, 2) if project_llm_calls else 0

    if project_errors == 0:
        project_status = "Saudável"
    elif project_successes > 0:
        project_status = "Atenção"
    else:
        project_status = "Com erro"

    rows = [
        {
            "Camada": "Projeto",
            "Nome": PROJECT_DISPLAY_NAME,
            "Função": "Sentrya Ops V2 completo",
            "Uso Principal": "Visão consolidada do projeto puxada do LangSmith",
            "Chamadas/Traces LangSmith": project_total,
            "Chamadas LLM": project_llm_calls,
            "Sucessos": project_successes,
            "Erros": project_errors,
            "Taxa de Erro": f"{project_error_rate}%",
            "Tokens Entrada": project_input_tokens,
            "Tokens Saída": project_output_tokens,
            "Tokens Totais": project_total_tokens,
            "Média Tokens/Chamada": project_avg_tokens,
            "Latência Média (s)": root_avg_latency,
            "Última Execução": root_last_execution,
            "Status Geral": project_status,
            "Fonte": "LangSmith read_project.run_count + root traces + LLM runs",
        }
    ]

    for model_name, model_info in EXPECTED_LLMS.items():
        model_runs = (
            llm_runs_df[llm_runs_df["model_name"] == model_name]
            if not llm_runs_df.empty
            else pd.DataFrame()
        )

        if model_runs.empty:
            rows.append(
                {
                    "Camada": "LLM",
                    "Nome": model_name,
                    "Função": model_info["role"],
                    "Uso Principal": model_info["purpose"],
                    "Chamadas/Traces LangSmith": 0,
                    "Chamadas LLM": 0,
                    "Sucessos": 0,
                    "Erros": 0,
                    "Taxa de Erro": "0%",
                    "Tokens Entrada": 0,
                    "Tokens Saída": 0,
                    "Tokens Totais": 0,
                    "Média Tokens/Chamada": 0,
                    "Latência Média (s)": 0,
                    "Última Execução": "Sem execução",
                    "Status Geral": "Sem execução",
                    "Fonte": "LangSmith LLM runs",
                }
            )
            continue

        model_calls = len(model_runs)
        model_errors = int(model_runs["has_error"].sum())
        model_successes = max(model_calls - model_errors, 0)
        model_error_rate = round((model_errors / model_calls) * 100, 2) if model_calls else 0

        model_input_tokens = int(model_runs["input_tokens"].sum())
        model_output_tokens = int(model_runs["output_tokens"].sum())
        model_total_tokens = int(model_runs["total_tokens"].sum())
        model_avg_tokens = round(model_total_tokens / model_calls, 2) if model_calls else 0

        model_avg_latency = (
            round(model_runs["latency_seconds"].dropna().mean(), 3)
            if model_runs["latency_seconds"].notna().any()
            else 0
        )

        model_last_execution = model_runs["start_time"].max()

        if model_errors == 0:
            model_status = "Saudável"
        elif model_successes > 0:
            model_status = "Atenção"
        else:
            model_status = "Com erro"

        rows.append(
            {
                "Camada": "LLM",
                "Nome": model_name,
                "Função": model_info["role"],
                "Uso Principal": model_info["purpose"],
                "Chamadas/Traces LangSmith": model_runs["trace_id"].nunique(),
                "Chamadas LLM": model_calls,
                "Sucessos": model_successes,
                "Erros": model_errors,
                "Taxa de Erro": f"{model_error_rate}%",
                "Tokens Entrada": model_input_tokens,
                "Tokens Saída": model_output_tokens,
                "Tokens Totais": model_total_tokens,
                "Média Tokens/Chamada": model_avg_tokens,
                "Latência Média (s)": model_avg_latency,
                "Última Execução": model_last_execution,
                "Status Geral": model_status,
                "Fonte": "LangSmith LLM runs",
            }
        )

    debug_info = {
        "project_found": dashboard_metrics["project_found"],
        "dashboard_count": dashboard_count,
        "dashboard_tokens": dashboard_tokens,
        "root_total": root_total,
        "root_errors": root_errors,
        "llm_runs": len(llm_runs_df) if not llm_runs_df.empty else 0,
        "updated_at": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    return pd.DataFrame(rows), debug_info


# Style table / Estiliza tabela
def style_langsmith_realtime_table(df: pd.DataFrame):
    return (
        df.style
        .hide(axis="index")
        .set_properties(
            **{
                "border": "1px solid #4b5563",
                "padding": "10px",
                "vertical-align": "middle",
                "font-size": "13px",
            }
        )
        .set_properties(
            subset=[
                "Camada",
                "Nome",
                "Função",
                "Chamadas/Traces LangSmith",
                "Chamadas LLM",
                "Sucessos",
                "Erros",
                "Taxa de Erro",
                "Tokens Entrada",
                "Tokens Saída",
                "Tokens Totais",
                "Média Tokens/Chamada",
                "Latência Média (s)",
                "Última Execução",
                "Status Geral",
            ],
            **{
                "text-align": "center",
                "white-space": "normal",
                "vertical-align": "middle",
            },
        )
        .set_properties(
            subset=["Uso Principal", "Fonte"],
            **{
                "text-align": "left",
                "white-space": "normal",
                "vertical-align": "middle",
                "min-width": "320px",
                "max-width": "520px",
            },
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("padding", "12px"),
                        ("text-align", "center"),
                        ("vertical-align", "middle"),
                        ("font-weight", "bold"),
                        ("background-color", "#111827"),
                        ("color", "#f9fafb"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("border-bottom", "2px solid #374151"),
                    ],
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                    ],
                },
            ]
        )
    )


# Refresh table from LangSmith / Atualiza tabela a partir do LangSmith
def refresh_langsmith_realtime_table():
    table_df, debug_info = build_langsmith_realtime_project_llm_table(LANGSMITH_PROJECT_NAME)

    print("LANGSMITH_REALTIME_PROJECT_AND_LLM_TABLE_OK")
    print("Projeto:", LANGSMITH_PROJECT_NAME)
    print("Atualizado em:", debug_info["updated_at"])
    print("Project found:", debug_info["project_found"])
    print("Dashboard count:", debug_info["dashboard_count"])
    print("Root traces fetched:", debug_info["root_total"])
    print("Root errors:", debug_info["root_errors"])
    print("LLM runs fetched:", debug_info["llm_runs"])
    print("Chamadas/Traces LangSmith:", int(table_df.loc[0, "Chamadas/Traces LangSmith"]))
    print("Sucessos:", int(table_df.loc[0, "Sucessos"]))
    print("Erros:", int(table_df.loc[0, "Erros"]))

    display(style_langsmith_realtime_table(table_df))

    return table_df, debug_info


# Run one real-time refresh / Executa uma atualização em tempo real
langsmith_realtime_df, langsmith_realtime_debug = refresh_langsmith_realtime_table()

In [ ]:
import time
from IPython.display import clear_output, display

# Auto-refresh LangSmith monitoring table / Atualiza automaticamente a tabela de monitoramento do LangSmith
AUTO_REFRESH_CYCLES = 5
REFRESH_SECONDS = 30

for cycle in range(AUTO_REFRESH_CYCLES):
    clear_output(wait=True)

    print(f"Atualização automática {cycle + 1}/{AUTO_REFRESH_CYCLES}")
    print(f"Intervalo entre atualizações: {REFRESH_SECONDS} segundos")
    print("-" * 80)

    # Refresh table from LangSmith / Atualiza a tabela a partir do LangSmith
    langsmith_realtime_df, langsmith_realtime_debug = refresh_langsmith_realtime_table()

    if cycle < AUTO_REFRESH_CYCLES - 1:
        print("-" * 80)
        print("Aguardando próxima atualização...")
        time.sleep(REFRESH_SECONDS)

print("AUTO_REFRESH_LANGSMITH_MONITORING_FINISHED")

In [ ]:
from pathlib import Path

# Create output folder / Cria a pasta de saída
output_dir = PROJECT_ROOT / "docs" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Define output file paths / Define os caminhos dos arquivos de saída
csv_path = output_dir / "sentrya_ops_v2_langsmith_project_llm_monitoring.csv"
html_path = output_dir / "sentrya_ops_v2_langsmith_project_llm_monitoring.html"

# Save dataframe as CSV / Salva o DataFrame como CSV
langsmith_realtime_df.to_csv(csv_path, index=False, encoding="utf-8")

# Save styled table as HTML / Salva a tabela estilizada como HTML
styled_html = style_langsmith_realtime_table(langsmith_realtime_df).to_html()
html_path.write_text(styled_html, encoding="utf-8")

print("LANGSMITH_MONITORING_TABLE_SAVED")
print("CSV:", csv_path)
print("HTML:", html_path)

In [ ]:
from pathlib import Path
import pandas as pd

# Select the latest LangSmith monitoring dataframe / Seleciona o DataFrame mais recente de monitoramento do LangSmith
source_df = None

if "langsmith_realtime_df" in globals():
    source_df = langsmith_realtime_df.copy()
elif "langsmith_truth_df" in globals():
    source_df = langsmith_truth_df.copy()
elif "project_llm_monitoring_df" in globals():
    source_df = project_llm_monitoring_df.copy()
else:
    raise NameError(
        "No LangSmith monitoring dataframe found. Run the LangSmith monitoring table cell first. / "
        "Nenhum DataFrame de monitoramento LangSmith encontrado. Rode primeiro a célula da tabela LangSmith."
    )


# Translate column names to English / Traduz os nomes das colunas para inglês
column_translation = {
    "Camada": "Layer",
    "Nome": "Name",
    "Função": "Agent Role",
    "Uso Principal": "Main Purpose",
    "Chamadas/Traces LangSmith": "LangSmith Calls/Traces",
    "Traces LangSmith": "LangSmith Traces",
    "Runs LangSmith": "LangSmith Runs",
    "Chamadas LLM": "LLM Calls",
    "Sucessos": "Successes",
    "Erros": "Errors",
    "Taxa de Erro": "Error Rate",
    "Tokens Entrada": "Input Tokens",
    "Tokens Saída": "Output Tokens",
    "Tokens Totais": "Total Tokens",
    "Média Tokens/Chamada": "Avg Tokens/Call",
    "Latência Média (s)": "Avg Latency (s)",
    "Última Execução": "Last Run",
    "Status Geral": "Overall Status",
    "Fonte": "Source",
}

langsmith_monitoring_en_df = source_df.rename(columns=column_translation)


# Translate row values to English / Traduz os valores das linhas para inglês
value_translation = {
    "Projeto": "Project",
    "Sentrya Ops V2 completo": "Complete Sentrya Ops V2",
    "Classificação rápida": "Fast Classification",
    "Agente principal": "Main Agent",
    "Raciocínio pesado": "Heavy Reasoning",
    "Comunicação geral": "General Communication",
    "Validação, pré-processamento e roteamento inicial": "Validation, preprocessing and initial routing",
    "Orquestração, decisão operacional, JSON estruturado e tool calling": "Orchestration, operational decision-making, structured JSON and tool calling",
    "Orquestração, decisão operacional, JSON e tool calling": "Orchestration, operational decision-making, JSON and tool calling",
    "Análise complexa, priorização e decisões críticas": "Complex analysis, prioritization and critical decisions",
    "Síntese, explicação natural e resposta final": "Synthesis, natural explanation and final response",
    "Visão consolidada do projeto puxada do LangSmith": "Consolidated project view pulled from LangSmith",
    "Visão consolidada do projeto usando contador real do LangSmith e métricas LLM": "Consolidated project view using LangSmith counter and LLM metrics",
    "Visão consolidada do projeto com dados reais do LangSmith": "Consolidated project view with real LangSmith data",
    "Saudável": "Healthy",
    "Atenção": "Needs Attention",
    "Com erro": "Error",
    "Sem execução": "No Runs",
    "LangSmith read_project.run_count + root traces + LLM runs": "LangSmith read_project.run_count + root traces + LLM runs",
    "LangSmith LLM runs": "LangSmith LLM runs",
}

for column in langsmith_monitoring_en_df.columns:
    langsmith_monitoring_en_df[column] = langsmith_monitoring_en_df[column].map(
        lambda value: value_translation.get(value, value)
    )


# Reorder columns in English / Reordena as colunas em inglês
preferred_columns = [
    "Layer",
    "Name",
    "Agent Role",
    "Main Purpose",
    "LangSmith Calls/Traces",
    "LLM Calls",
    "Successes",
    "Errors",
    "Error Rate",
    "Input Tokens",
    "Output Tokens",
    "Total Tokens",
    "Avg Tokens/Call",
    "Avg Latency (s)",
    "Last Run",
    "Overall Status",
    "Source",
]

existing_columns = [
    column for column in preferred_columns
    if column in langsmith_monitoring_en_df.columns
]

remaining_columns = [
    column for column in langsmith_monitoring_en_df.columns
    if column not in existing_columns
]

langsmith_monitoring_en_df = langsmith_monitoring_en_df[
    existing_columns + remaining_columns
]


# Style English LangSmith monitoring table / Estiliza a tabela LangSmith em inglês
def style_langsmith_monitoring_en_table(df: pd.DataFrame):
    return (
        df.style
        .hide(axis="index")
        .set_properties(
            **{
                "border": "1px solid #4b5563",
                "padding": "10px",
                "vertical-align": "middle",
                "font-size": "13px",
            }
        )
        .set_properties(
            subset=[
                column for column in [
                    "Layer",
                    "Name",
                    "Agent Role",
                    "LangSmith Calls/Traces",
                    "LLM Calls",
                    "Successes",
                    "Errors",
                    "Error Rate",
                    "Input Tokens",
                    "Output Tokens",
                    "Total Tokens",
                    "Avg Tokens/Call",
                    "Avg Latency (s)",
                    "Last Run",
                    "Overall Status",
                ]
                if column in df.columns
            ],
            **{
                "text-align": "center",
                "white-space": "normal",
                "vertical-align": "middle",
            },
        )
        .set_properties(
            subset=[
                column for column in ["Main Purpose", "Source"]
                if column in df.columns
            ],
            **{
                "text-align": "left",
                "white-space": "normal",
                "vertical-align": "middle",
                "min-width": "320px",
                "max-width": "520px",
            },
        )
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("padding", "12px"),
                        ("text-align", "center"),
                        ("vertical-align", "middle"),
                        ("font-weight", "bold"),
                        ("background-color", "#111827"),
                        ("color", "#f9fafb"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("border", "1px solid #4b5563"),
                        ("border-bottom", "2px solid #374151"),
                    ],
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                    ],
                },
            ]
        )
    )


# Save English monitoring table / Salva a tabela de monitoramento em inglês
output_dir = PROJECT_ROOT / "docs" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path_en = output_dir / "sentrya_ops_v2_langsmith_project_llm_monitoring_en.csv"
html_path_en = output_dir / "sentrya_ops_v2_langsmith_project_llm_monitoring_en.html"

langsmith_monitoring_en_df.to_csv(csv_path_en, index=False, encoding="utf-8")

styled_html_en = style_langsmith_monitoring_en_table(langsmith_monitoring_en_df).to_html()
html_path_en.write_text(styled_html_en, encoding="utf-8")


print("LANGSMITH_MONITORING_TABLE_ENGLISH_OK")
print("CSV:", csv_path_en)
print("HTML:", html_path_en)
print("Rows:", len(langsmith_monitoring_en_df))

style_langsmith_monitoring_en_table(langsmith_monitoring_en_df)